# Cell 1 — Load the Sequence-Variation Feature Engineering Cohort

In [12]:
from google.colab import drive

import os
import pandas as pd


# =========================================================
# FILE 06 — CELL 1
# Load and verify sequence-variation input cohort
# =========================================================


# =========================================================
# 1. Fixed Google Drive mount
# =========================================================

DRIVE_MOUNT = (
    "/content/drive_real"
)


# Mount only if MyDrive is not currently available
if not os.path.isdir(
    os.path.join(
        DRIVE_MOUNT,
        "MyDrive"
    )
):

    os.makedirs(
        DRIVE_MOUNT,
        exist_ok=True
    )

    # Mountpoint must be empty if not already mounted
    if os.listdir(
        DRIVE_MOUNT
    ):

        raise RuntimeError(
            "/content/drive_real is non-empty but "
            "Google Drive is not mounted there. "
            "Restart the runtime before continuing."
        )

    drive.mount(
        DRIVE_MOUNT
    )

else:

    print(
        "Google Drive already mounted at "
        "/content/drive_real."
    )


# =========================================================
# 2. Project root
# =========================================================

MYDRIVE = os.path.join(
    DRIVE_MOUNT,
    "MyDrive"
)

DRIVE_ROOT = os.path.join(
    MYDRIVE,
    "AMR-Genome-ML"
)


assert os.path.isdir(
    DRIVE_ROOT
), (
    f"Project directory not found:\n"
    f"{DRIVE_ROOT}"
)


print()

print(
    "PASS: Google Drive available."
)

print(
    f"Drive mount : {DRIVE_MOUNT}"
)

print(
    f"Project root: {DRIVE_ROOT}"
)


# =========================================================
# 3. Upstream finalized inputs
# =========================================================

PANGENOME_MANIFEST_PATH = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "pangenome",
    "pangenome_analysis_manifest.csv"
)

PANGENOME_ROW_INDEX_PATH = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "pangenome",
    "X_pangenome_row_index.csv"
)

FILE05_SUMMARY_PATH = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "file05_pangenome_feature_engineering_summary.csv"
)

X_KNOWN_AMR_PATH = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "known_amr",
    "X_known_amr_final.csv"
)


# =========================================================
# 4. File 06 directories
# =========================================================

SEQUENCE_DIR = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "sequence_variation"
)

SEQUENCE_CHECKPOINT_DIR = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "sequence_variation"
)

SEQUENCE_TMP_DIR = (
    "/content/sequence_variation_work"
)


for path in [
    SEQUENCE_DIR,
    SEQUENCE_CHECKPOINT_DIR,
    SEQUENCE_TMP_DIR
]:

    os.makedirs(
        path,
        exist_ok=True
    )


# =========================================================
# 5. Verify upstream files
# =========================================================

required_files = [
    PANGENOME_MANIFEST_PATH,
    PANGENOME_ROW_INDEX_PATH,
    FILE05_SUMMARY_PATH,
    X_KNOWN_AMR_PATH
]


missing_files = [
    path
    for path in required_files
    if not os.path.exists(
        path
    )
]


if missing_files:

    raise FileNotFoundError(
        "Missing required upstream files:\n"
        +
        "\n".join(
            missing_files
        )
    )


# =========================================================
# 6. Load finalized common cohort
# =========================================================

df_sequence_input = pd.read_csv(
    PANGENOME_MANIFEST_PATH,
    dtype={
        "Genome ID": str,
        "Taxon ID": str
    }
)


df_sequence_input[
    "Genome ID"
] = (
    df_sequence_input[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


assert len(
    df_sequence_input
) == 4227


assert (
    df_sequence_input[
        "Genome ID"
    ]
    .nunique()
    == 4227
)


# =========================================================
# 7. Remap persisted FASTA paths to current Drive mount
# =========================================================

def remap_drive_path(
    path
):

    path = str(
        path
    ).strip()


    marker = (
        "/MyDrive/"
    )


    if marker in path:

        relative_path = path.split(
            marker,
            1
        )[1]

        return os.path.join(
            MYDRIVE,
            relative_path
        )


    return path


old_fasta_paths = (
    df_sequence_input[
        "FASTA Path"
    ]
    .astype(str)
    .copy()
)


df_sequence_input[
    "FASTA Path"
] = (
    df_sequence_input[
        "FASTA Path"
    ]
    .apply(
        remap_drive_path
    )
)


remapped_count = int(
    (
        old_fasta_paths
        !=
        df_sequence_input[
            "FASTA Path"
        ]
    ).sum()
)


# =========================================================
# 8. Verify cohort order against File 05
# =========================================================

df_pangenome_rows = pd.read_csv(
    PANGENOME_ROW_INDEX_PATH,
    dtype={
        "Genome ID": str
    }
)


df_pangenome_rows[
    "Genome ID"
] = (
    df_pangenome_rows[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


assert len(
    df_pangenome_rows
) == 4227


assert (
    df_sequence_input[
        "Genome ID"
    ]
    .tolist()
    ==
    df_pangenome_rows[
        "Genome ID"
    ]
    .tolist()
)


# =========================================================
# 9. Verify all FASTA files
# =========================================================

df_sequence_input[
    "FASTA Exists"
] = (
    df_sequence_input[
        "FASTA Path"
    ]
    .astype(str)
    .apply(
        os.path.exists
    )
)


missing_fasta = (
    df_sequence_input[
        ~df_sequence_input[
            "FASTA Exists"
        ]
    ]
)


assert missing_fasta.empty, (
    f"{len(missing_fasta):,} FASTA files "
    f"are missing."
)


compressed_fasta_count = int(
    df_sequence_input[
        "FASTA Path"
    ]
    .astype(str)
    .str.endswith(
        ".gz"
    )
    .sum()
)


assert compressed_fasta_count == 4227


# =========================================================
# 10. Verify common cohort with X_known_amr
# =========================================================

df_known_amr_ids = pd.read_csv(
    X_KNOWN_AMR_PATH,
    usecols=[
        "Genome ID"
    ],
    dtype={
        "Genome ID": str
    }
)


df_known_amr_ids[
    "Genome ID"
] = (
    df_known_amr_ids[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


known_amr_ids = set(
    df_known_amr_ids[
        "Genome ID"
    ]
)


sequence_ids = set(
    df_sequence_input[
        "Genome ID"
    ]
)


common_known_amr_n = len(
    sequence_ids
    &
    known_amr_ids
)


assert common_known_amr_n == 4227


# =========================================================
# 11. Audit summaries
# =========================================================

phenotype_counts = (
    df_sequence_input[
        "Resistant Phenotype"
    ]
    .value_counts()
)


fasta_sizes = (
    df_sequence_input[
        "FASTA Path"
    ]
    .apply(
        os.path.getsize
    )
)


total_fasta_gib = (
    fasta_sizes.sum()
    / (1024 ** 3)
)


# =========================================================
# 12. Final report
# =========================================================

print()
print("=" * 70)
print("FILE 06 — SEQUENCE VARIATION FEATURE ENGINEERING")
print("=" * 70)

print(
    f"Common-cohort genomes      : "
    f"{len(df_sequence_input):,}"
)

print(
    f"Unique Genome IDs          : "
    f"{df_sequence_input['Genome ID'].nunique():,}"
)

print(
    f"Available FASTA files      : "
    f"{df_sequence_input['FASTA Exists'].sum():,}"
)

print(
    f"Compressed FASTA files     : "
    f"{compressed_fasta_count:,}"
)

print(
    f"Compressed FASTA storage   : "
    f"{total_fasta_gib:.2f} GiB"
)

print(
    f"FASTA paths remapped       : "
    f"{remapped_count:,}"
)

print(
    f"Common with X_known_amr    : "
    f"{common_known_amr_n:,}"
)

print()

print(
    "Phenotype distribution:"
)

print(
    phenotype_counts
)

print()

print(
    "Sequence-variation output directory:"
)

print(
    SEQUENCE_DIR
)

print()

print(
    "PASS: File 06 input cohort is complete "
    "and all 4,227 compressed genome assemblies "
    "are available from the verified Drive mount."
)

Google Drive already mounted at /content/drive_real.

PASS: Google Drive available.
Drive mount : /content/drive_real
Project root: /content/drive_real/MyDrive/AMR-Genome-ML

FILE 06 — SEQUENCE VARIATION FEATURE ENGINEERING
Common-cohort genomes      : 4,227
Unique Genome IDs          : 4,227
Available FASTA files      : 4,227
Compressed FASTA files     : 4,227
Compressed FASTA storage   : 7.70 GiB
FASTA paths remapped       : 4,227
Common with X_known_amr    : 4,227

Phenotype distribution:
Resistant Phenotype
Susceptible    2626
Resistant      1601
Name: count, dtype: int64

Sequence-variation output directory:
/content/drive_real/MyDrive/AMR-Genome-ML/data/features/sequence_variation

PASS: File 06 input cohort is complete and all 4,227 compressed genome assemblies are available from the verified Drive mount.


# Cell 2 — Select the Sequence-Variation Strategy and Pilot Cohort

In [ ]:
import os
import shutil
import platform
import numpy as np
import pandas as pd


# =========================================================
# 1. Sequence-variation strategy
# =========================================================

SEQUENCE_STRATEGY = (
    "Reference-free unitig presence/absence"
)

UNITIG_K = 31

UNITIG_ENCODING = (
    "Binary presence/absence"
)

UNITIG_PILOT_N = 64


# =========================================================
# 2. File 06 persistent paths
# =========================================================

SEQUENCE_STRATEGY_PATH = os.path.join(
    SEQUENCE_DIR,
    "sequence_variation_strategy_decision.csv"
)

UNITIG_PILOT_MANIFEST_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_pilot_manifest.csv"
)

FILE06_RESOURCE_AUDIT_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "file06_runtime_resource_audit.csv"
)


# =========================================================
# 3. Runtime resource audit
# =========================================================

cpu_count = os.cpu_count()


# RAM from Linux /proc
mem_total_kb = None

with open(
    "/proc/meminfo",
    "r"
) as handle:

    for line in handle:

        if line.startswith(
            "MemTotal:"
        ):

            mem_total_kb = int(
                line.split()[1]
            )

            break


assert mem_total_kb is not None


ram_gib = (
    mem_total_kb
    / 1024
    / 1024
)


disk_usage = shutil.disk_usage(
    "/content"
)


disk_total_gib = (
    disk_usage.total
    / (1024 ** 3)
)

disk_free_gib = (
    disk_usage.free
    / (1024 ** 3)
)


df_resource_audit = pd.DataFrame(
    [
        {
            "CPU Count":
                cpu_count,

            "RAM GiB":
                ram_gib,

            "Local Disk Total GiB":
                disk_total_gib,

            "Local Disk Free GiB":
                disk_free_gib,

            "Python":
                platform.python_version(),

            "Platform":
                platform.platform()
        }
    ]
)


df_resource_audit.to_csv(
    FILE06_RESOURCE_AUDIT_PATH,
    index=False
)


# =========================================================
# 4. Candidate representation decision record
# =========================================================

df_strategy_candidates = pd.DataFrame(
    [
        {
            "Candidate":
                "Core-genome SNP matrix",

            "Reference Free":
                False,

            "Captures Within-Gene Variation":
                True,

            "Captures Accessory Sequence Variation":
                False,

            "Primary Choice":
                False,

            "Decision":
                (
                    "Not primary; requires reference/core alignment "
                    "and represents only the alignable core genome."
                )
        },

        {
            "Candidate":
                "Raw canonical k-mers",

            "Reference Free":
                True,

            "Captures Within-Gene Variation":
                True,

            "Captures Accessory Sequence Variation":
                True,

            "Primary Choice":
                False,

            "Decision":
                (
                    "Not primary; raw k-mer space is highly redundant "
                    "and potentially extremely high-dimensional."
                )
        },

        {
            "Candidate":
                "Compacted de Bruijn graph unitigs",

            "Reference Free":
                True,

            "Captures Within-Gene Variation":
                True,

            "Captures Accessory Sequence Variation":
                True,

            "Primary Choice":
                True,

            "Decision":
                (
                    "Primary sequence-level representation; "
                    "retains exact nucleotide sequence elements while "
                    "reducing redundancy relative to raw k-mers."
                )
        }
    ]
)


# =========================================================
# 5. Final strategy record
# =========================================================

df_sequence_strategy = pd.DataFrame(
    [
        {
            "Representation":
                SEQUENCE_STRATEGY,

            "Graph Type":
                "Compacted coloured de Bruijn graph",

            "Planned Tool":
                "unitig-caller / Bifrost",

            "K-mer Size":
                UNITIG_K,

            "Encoding":
                UNITIG_ENCODING,

            "Study Cohort":
                len(
                    df_sequence_input
                ),

            "Pilot Genomes":
                UNITIG_PILOT_N,

            "Phenotype Used for Strategy":
                False,

            "Phenotype Used for Pilot Selection":
                False,

            "Full Cohort Run Approved":
                False
        }
    ]
)


df_sequence_strategy.to_csv(
    SEQUENCE_STRATEGY_PATH,
    index=False
)


# =========================================================
# 6. Calculate compressed FASTA size
# =========================================================

df_unitig_pilot_source = (
    df_sequence_input[
        [
            "Genome ID",
            "FASTA Path"
        ]
    ]
    .copy()
)


df_unitig_pilot_source[
    "FASTA Size Bytes"
] = (
    df_unitig_pilot_source[
        "FASTA Path"
    ]
    .astype(str)
    .apply(
        os.path.getsize
    )
)


df_unitig_pilot_source[
    "FASTA Size MiB"
] = (
    df_unitig_pilot_source[
        "FASTA Size Bytes"
    ]
    / (1024 ** 2)
)


# =========================================================
# 7. Deterministic phenotype-independent pilot selection
#
# Sort by compressed FASTA size and select evenly spaced
# ranks across the full cohort.
# =========================================================

df_unitig_pilot_source = (
    df_unitig_pilot_source
    .sort_values(
        [
            "FASTA Size Bytes",
            "Genome ID"
        ]
    )
    .reset_index(
        drop=True
    )
)


pilot_positions = np.linspace(
    0,
    len(
        df_unitig_pilot_source
    ) - 1,
    UNITIG_PILOT_N,
    dtype=int
)


assert len(
    np.unique(
        pilot_positions
    )
) == UNITIG_PILOT_N


df_unitig_pilot = (
    df_unitig_pilot_source
    .iloc[
        pilot_positions
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


df_unitig_pilot[
    "Pilot Index"
] = np.arange(
    1,
    UNITIG_PILOT_N + 1
)


df_unitig_pilot = (
    df_unitig_pilot[
        [
            "Pilot Index",
            "Genome ID",
            "FASTA Path",
            "FASTA Size Bytes",
            "FASTA Size MiB"
        ]
    ]
)


# =========================================================
# 8. Pilot integrity
# =========================================================

assert len(
    df_unitig_pilot
) == UNITIG_PILOT_N


assert (
    df_unitig_pilot[
        "Genome ID"
    ]
    .nunique()
    == UNITIG_PILOT_N
)


assert (
    df_unitig_pilot[
        "FASTA Path"
    ]
    .apply(
        os.path.exists
    )
    .all()
)


# No phenotype column is present in the pilot-selection table.
assert (
    "Resistant Phenotype"
    not in df_unitig_pilot.columns
)


# =========================================================
# 9. Save pilot manifest
# =========================================================

df_unitig_pilot.to_csv(
    UNITIG_PILOT_MANIFEST_PATH,
    index=False
)


# =========================================================
# 10. Report
# =========================================================

print("=" * 70)
print("SEQUENCE-VARIATION STRATEGY")
print("=" * 70)

print(
    f"Primary representation : "
    f"{SEQUENCE_STRATEGY}"
)

print(
    f"Planned tool           : "
    f"unitig-caller / Bifrost"
)

print(
    f"k-mer size             : "
    f"{UNITIG_K}"
)

print(
    f"Encoding               : "
    f"{UNITIG_ENCODING}"
)

print(
    f"Full cohort            : "
    f"{len(df_sequence_input):,}"
)

print(
    f"Pilot cohort           : "
    f"{len(df_unitig_pilot):,}"
)

print()

print(
    "Runtime resources:"
)

display(
    df_resource_audit
)

print()

print(
    "Candidate strategy comparison:"
)

display(
    df_strategy_candidates
)

print()

print(
    "Pilot FASTA-size distribution:"
)

display(
    df_unitig_pilot[
        "FASTA Size MiB"
    ]
    .describe()
    .to_frame(
        name="Compressed FASTA MiB"
    )
)

print()

print(
    "Pilot genomes:"
)

display(
    df_unitig_pilot
    .head(10)
)

print()

print(
    "Strategy decision saved to:"
)

print(
    SEQUENCE_STRATEGY_PATH
)

print()

print(
    "Pilot manifest saved to:"
)

print(
    UNITIG_PILOT_MANIFEST_PATH
)

print()

print(
    "PASS: Reference-free unitig presence/absence "
    "was selected as the primary sequence-variation "
    "strategy, pending pilot resource benchmarking."
)

SEQUENCE-VARIATION STRATEGY
Primary representation : Reference-free unitig presence/absence
Planned tool           : unitig-caller / Bifrost
k-mer size             : 31
Encoding               : Binary presence/absence
Full cohort            : 4,227
Pilot cohort           : 64

Runtime resources:


,CPU Count,RAM GiB,Local Disk Total GiB,Local Disk Free GiB,Python,Platform
0,2,12.67141,107.715084,87.366856,3.13.15,Linux-6.6.122+-x86_64-with-glibc2.39



Candidate strategy comparison:


,Candidate,Reference Free,Captures Within-Gene Variation,Captures Accessory Sequence Variation,Primary Choice,Decision
0,Core-genome SNP matrix,False,True,False,False,Not primary; requires reference/core alignment...
1,Raw canonical k-mers,True,True,True,False,Not primary; raw k-mer space is highly redunda...
2,Compacted de Bruijn graph unitigs,True,True,True,True,Primary sequence-level representation; retains...



Pilot FASTA-size distribution:


,Compressed FASTA MiB
count,64.000000
mean,1.852208
std,0.164719
min,0.676970
25%,1.827798
50%,1.860560
75%,1.896842
max,2.227835



Pilot genomes:


,Pilot Index,Genome ID,FASTA Path,FASTA Size Bytes,FASTA Size MiB
0,1,573.31059,/content/drive/MyDrive/AMR-Genome-ML/data/geno...,709854,0.676970
1,2,573.66982,/content/drive/MyDrive/AMR-Genome-ML/data/geno...,1822594,1.738161
2,3,573.66666,/content/drive/MyDrive/AMR-Genome-ML/data/geno...,1848461,1.762830
3,4,573.67107,/content/drive/MyDrive/AMR-Genome-ML/data/geno...,1862794,1.776499
4,5,573.15539,/content/drive/MyDrive/AMR-Genome-ML/data/geno...,1873289,1.786508
5,6,573.66794,/content/drive/MyDrive/AMR-Genome-ML/data/geno...,1880217,1.793115
6,7,573.14448,/content/drive/MyDrive/AMR-Genome-ML/data/geno...,1886356,1.798969
7,8,573.22555,/content/drive/MyDrive/AMR-Genome-ML/data/geno...,1891281,1.803666
8,9,573.5702,/content/drive/MyDrive/AMR-Genome-ML/data/geno...,1895496,1.807686
9,10,573.67323,/content/drive/MyDrive/AMR-Genome-ML/data/geno...,1898654,1.810698



Strategy decision saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/sequence_variation/sequence_variation_strategy_decision.csv

Pilot manifest saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/sequence_variation/unitig_pilot_manifest.csv

PASS: Reference-free unitig presence/absence was selected as the primary sequence-variation strategy, pending pilot resource benchmarking.


# Cell 3 — Install and Verify the Unitig-Calling Toolchain

In [13]:
import os
import shutil
import subprocess
import textwrap


# =========================================================
# 1. Local tool paths
# =========================================================

MICROMAMBA_BIN = (
    "/content/bin/micromamba"
)

MAMBA_ROOT_PREFIX = (
    "/content/micromamba"
)

UNITIG_ENV = (
    "/content/unitig_env"
)

UNITIG_CALLER_BIN = os.path.join(
    UNITIG_ENV,
    "bin",
    "unitig-caller"
)


os.makedirs(
    "/content/bin",
    exist_ok=True
)


# =========================================================
# 2. Helper for visible shell commands
# =========================================================

def run_command(
    command,
    env=None
):

    print()
    print(
        "$",
        " ".join(command)
    )

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env
    )

    print(
        result.stdout
    )

    if result.returncode != 0:

        raise RuntimeError(
            f"Command failed with exit code "
            f"{result.returncode}"
        )

    return result.stdout


# =========================================================
# 3. Install micromamba if needed
# =========================================================

if not os.path.exists(
    MICROMAMBA_BIN
):

    print(
        "Installing micromamba..."
    )

    subprocess.run(
        [
            "bash",
            "-lc",
            (
                "cd /content && "
                "curl -Ls "
                "https://micro.mamba.pm/api/micromamba/"
                "linux-64/latest "
                "| tar -xj bin/micromamba"
            )
        ],
        check=True
    )


assert os.path.exists(
    MICROMAMBA_BIN
)


# =========================================================
# 4. Micromamba environment variables
# =========================================================

mamba_env = os.environ.copy()

mamba_env[
    "MAMBA_ROOT_PREFIX"
] = MAMBA_ROOT_PREFIX


# =========================================================
# 5. Create isolated unitig-caller environment
# =========================================================

if not os.path.exists(
    UNITIG_CALLER_BIN
):

    print()
    print(
        "Creating isolated unitig-caller environment..."
    )

    run_command(
        [
            MICROMAMBA_BIN,
            "create",
            "-y",
            "-p",
            UNITIG_ENV,
            "-c",
            "conda-forge",
            "-c",
            "bioconda",
            "--strict-channel-priority",
            "unitig-caller=1.3.2"
        ],
        env=mamba_env
    )

else:

    print(
        "Existing unitig-caller environment found; "
        "installation skipped."
    )


# =========================================================
# 6. Verify executables
# =========================================================

assert os.path.exists(
    UNITIG_CALLER_BIN
), (
    "unitig-caller executable was not found."
)


# =========================================================
# 7. Verify unitig-caller version
# =========================================================

unitig_version_output = run_command(
    [
        UNITIG_CALLER_BIN,
        "--version"
    ],
    env=mamba_env
)


# =========================================================
# 8. Verify command-line interface
# =========================================================

help_output = run_command(
    [
        UNITIG_CALLER_BIN,
        "--help"
    ],
    env=mamba_env
)


required_cli_options = [
    "--call",
    "--refs",
    "--out",
    "--kmer",
    "--threads"
]


for option in required_cli_options:

    assert option in help_output, (
        f"Expected CLI option missing: {option}"
    )


# =========================================================
# 9. Check Bifrost dependency presence
# =========================================================

candidate_bifrost_bins = [
    os.path.join(
        UNITIG_ENV,
        "bin",
        "Bifrost"
    ),
    os.path.join(
        UNITIG_ENV,
        "bin",
        "bifrost"
    )
]


BIFROST_BIN = next(
    (
        path
        for path in candidate_bifrost_bins
        if os.path.exists(path)
    ),
    None
)


# Some unitig-caller builds access Bifrost through
# installed libraries rather than requiring direct CLI use.
bifrost_cli_found = (
    BIFROST_BIN is not None
)


# =========================================================
# 10. Environment package record
# =========================================================

package_list_output = run_command(
    [
        MICROMAMBA_BIN,
        "list",
        "-p",
        UNITIG_ENV
    ],
    env=mamba_env
)


assert (
    "unitig-caller"
    in package_list_output
)


assert (
    "bifrost"
    in package_list_output.lower()
)


# =========================================================
# 11. Save runtime verification
# =========================================================

TOOLCHAIN_AUDIT_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_toolchain_verified.txt"
)


with open(
    TOOLCHAIN_AUDIT_PATH,
    "w"
) as handle:

    handle.write(
        "UNITIG-CALLER TOOLCHAIN VERIFICATION\n"
    )

    handle.write(
        "=" * 60
        + "\n"
    )

    handle.write(
        f"Environment: {UNITIG_ENV}\n"
    )

    handle.write(
        f"unitig-caller: {UNITIG_CALLER_BIN}\n"
    )

    handle.write(
        f"Bifrost CLI found: {bifrost_cli_found}\n"
    )

    handle.write(
        f"Bifrost CLI path: {BIFROST_BIN}\n"
    )

    handle.write(
        "\nVersion output:\n"
    )

    handle.write(
        unitig_version_output
    )


# =========================================================
# 12. Final report
# =========================================================

print()
print("=" * 70)
print("UNITIG TOOLCHAIN VERIFICATION")
print("=" * 70)

print(
    f"Environment       : "
    f"{UNITIG_ENV}"
)

print(
    f"unitig-caller     : "
    f"{UNITIG_CALLER_BIN}"
)

print(
    f"Bifrost CLI found : "
    f"{bifrost_cli_found}"
)

print(
    f"Bifrost CLI       : "
    f"{BIFROST_BIN}"
)

print()

print(
    "unitig-caller version:"
)

print(
    unitig_version_output.strip()
)

print()

print(
    "Toolchain audit saved to:"
)

print(
    TOOLCHAIN_AUDIT_PATH
)

print()

print(
    "PASS: unitig-caller and its Bifrost "
    "dependencies are installed and ready "
    "for the 64-genome pilot benchmark."
)

Existing unitig-caller environment found; installation skipped.

$ /content/unitig_env/bin/unitig-caller --version
unitig-caller 1.3.2


$ /content/unitig_env/bin/unitig-caller --help
usage: unitig-caller [-h] (--call | --query | --simple) [--refs REFS]
                     [--reads READS] [--graph GRAPH] [--colours COLOURS]
                     [--unitigs UNITIGS] [--pyseer] [--rtab] [--out OUT]
                     [--kmer KMER] [--write-graph] [--no-save-idx]
                     [--threads THREADS] [--version]

Call unitigs in a population dataset

options:
  -h, --help         show this help message and exit

Mode of operation:
  --call             Build a DBG and call colours of unitigs within
  --query            Query unitig colours in reference genomes/DBG
  --simple           Use FM-index to make calls

Unitig-caller input/output:
  --refs REFS        Ref file to used to build DBG or use with --simple
  --reads READS      Read file to used to build DBG
  --graph GRAPH      Ex

# Cell 4 — Benchmark Unitig Calling on the 64-Genome Pilot

In [ ]:
import os
import re
import gzip
import glob
import time
import shutil
import subprocess

from collections import Counter

import numpy as np
import pandas as pd


# =========================================================
# 1. Persistent / local paths
# =========================================================

UNITIG_ENV = (
    "/content/unitig_env"
)

UNITIG_CALLER_BIN = os.path.join(
    UNITIG_ENV,
    "bin",
    "unitig-caller"
)

UNITIG_PILOT_MANIFEST_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_pilot_manifest.csv"
)

PILOT_ROOT = os.path.join(
    SEQUENCE_TMP_DIR,
    "unitig_pilot64"
)

PILOT_FASTA_DIR = os.path.join(
    PILOT_ROOT,
    "assemblies"
)

PILOT_OUTPUT_DIR = os.path.join(
    PILOT_ROOT,
    "output"
)

PILOT_REFS_PATH = os.path.join(
    PILOT_ROOT,
    "refs.txt"
)

PILOT_OUTPUT_PREFIX = os.path.join(
    PILOT_OUTPUT_DIR,
    "pilot64_k31"
)


# Persistent benchmark outputs
PILOT_BENCHMARK_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_pilot64_benchmark.csv"
)

PILOT_PREVALENCE_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_pilot64_prevalence_spectrum.csv"
)

PILOT_OUTPUT_INVENTORY_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_pilot64_output_inventory.csv"
)

PILOT_LOG_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_pilot64_run.log"
)


# =========================================================
# 2. Tool / parameter configuration
# =========================================================

UNITIG_K = 31

UNITIG_THREADS = 2

PILOT_N = 64


assert os.path.exists(
    UNITIG_CALLER_BIN
), (
    "unitig-caller environment is missing. "
    "Run Cell 3 first."
)


# =========================================================
# 3. Reload pilot manifest
# =========================================================

df_pilot_04 = pd.read_csv(
    UNITIG_PILOT_MANIFEST_PATH,
    dtype={
        "Genome ID": str
    }
)


df_pilot_04[
    "Genome ID"
] = (
    df_pilot_04[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


assert len(
    df_pilot_04
) == PILOT_N


assert (
    df_pilot_04[
        "Genome ID"
    ]
    .nunique()
    == PILOT_N
)


assert (
    df_pilot_04[
        "FASTA Path"
    ]
    .apply(
        os.path.exists
    )
    .all()
)


# =========================================================
# 4. Check local disk before staging
# =========================================================

disk_before = shutil.disk_usage(
    "/content"
)


disk_free_before_gib = (
    disk_before.free
    / (1024 ** 3)
)


print("=" * 70)
print("64-GENOME UNITIG PILOT")
print("=" * 70)

print(
    f"Pilot genomes        : "
    f"{PILOT_N}"
)

print(
    f"k-mer size           : "
    f"{UNITIG_K}"
)

print(
    f"Threads              : "
    f"{UNITIG_THREADS}"
)

print(
    f"Local free disk      : "
    f"{disk_free_before_gib:.2f} GiB"
)


assert disk_free_before_gib >= 15, (
    "Less than 15 GiB local disk is available; "
    "pilot aborted before staging."
)


# =========================================================
# 5. Create clean local pilot workspace
# =========================================================

# Pilot results are disposable local benchmark data.
# Persistent summaries remain on Drive.
if os.path.exists(
    PILOT_ROOT
):

    shutil.rmtree(
        PILOT_ROOT
    )


os.makedirs(
    PILOT_FASTA_DIR,
    exist_ok=True
)

os.makedirs(
    PILOT_OUTPUT_DIR,
    exist_ok=True
)


# =========================================================
# 6. Stage the 64 assemblies as uncompressed FASTA
# =========================================================

staging_records = []

staging_start = time.time()


for i, row in df_pilot_04.iterrows():

    genome_id = row[
        "Genome ID"
    ]

    source_path = str(
        row[
            "FASTA Path"
        ]
    )


    destination_path = os.path.join(
        PILOT_FASTA_DIR,
        f"{genome_id}.fasta"
    )


    # -----------------------------------------------------
    # Stream decompression / copying
    # -----------------------------------------------------

    if source_path.endswith(
        ".gz"
    ):

        with gzip.open(
            source_path,
            "rb"
        ) as src, open(
            destination_path,
            "wb"
        ) as dst:

            shutil.copyfileobj(
                src,
                dst,
                length=1024 * 1024
            )

    else:

        shutil.copyfile(
            source_path,
            destination_path
        )


    output_size = os.path.getsize(
        destination_path
    )


    assert output_size > 0


    staging_records.append(
        {
            "Genome ID":
                genome_id,

            "Source FASTA":
                source_path,

            "Local FASTA":
                destination_path,

            "Uncompressed Bytes":
                output_size
        }
    )


    if (
        (i + 1) % 16 == 0
        or (i + 1) == PILOT_N
    ):

        print(
            f"Staged "
            f"{i + 1:02d}/{PILOT_N} assemblies"
        )


df_pilot_staging = pd.DataFrame(
    staging_records
)


staging_elapsed = (
    time.time()
    - staging_start
)


staged_gib = (
    df_pilot_staging[
        "Uncompressed Bytes"
    ]
    .sum()
    / (1024 ** 3)
)


# =========================================================
# 7. Generate refs.txt
#
# unitig-caller expects one absolute assembly path per line,
# without a header.
# =========================================================

with open(
    PILOT_REFS_PATH,
    "w"
) as handle:

    for path in df_pilot_staging[
        "Local FASTA"
    ]:

        handle.write(
            os.path.abspath(
                path
            )
            + "\n"
        )


with open(
    PILOT_REFS_PATH,
    "r"
) as handle:

    refs_lines = [
        line.strip()
        for line in handle
        if line.strip()
    ]


assert len(
    refs_lines
) == PILOT_N


assert all(
    os.path.exists(
        path
    )
    for path in refs_lines
)


print()
print(
    f"Uncompressed pilot FASTA: "
    f"{staged_gib:.3f} GiB"
)

print(
    f"Staging runtime          : "
    f"{staging_elapsed:.1f} s"
)


# =========================================================
# 8. Run unitig-caller with GNU time resource profiling
# =========================================================

command = [
    "/usr/bin/time",
    "-v",
    UNITIG_CALLER_BIN,
    "--call",
    "--refs",
    PILOT_REFS_PATH,
    "--out",
    PILOT_OUTPUT_PREFIX,
    "--kmer",
    str(
        UNITIG_K
    ),
    "--threads",
    str(
        UNITIG_THREADS
    ),
    "--pyseer",
    "--write-graph"
]


print()
print("=" * 70)
print("RUNNING UNITIG-CALLER")
print("=" * 70)

print(
    " ".join(
        command
    )
)

print()


run_start = time.time()


result = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)


wall_seconds = (
    time.time()
    - run_start
)


# Save full software log persistently
with open(
    PILOT_LOG_PATH,
    "w"
) as handle:

    handle.write(
        result.stdout
    )


print(
    result.stdout
)


if result.returncode != 0:

    raise RuntimeError(
        "unitig-caller pilot failed. "
        f"See log: {PILOT_LOG_PATH}"
    )


# =========================================================
# 9. Parse peak memory from /usr/bin/time
# =========================================================

rss_match = re.search(
    r"Maximum resident set size \(kbytes\):\s*(\d+)",
    result.stdout
)


if rss_match:

    peak_rss_kb = int(
        rss_match.group(1)
    )

    peak_rss_gib = (
        peak_rss_kb
        / 1024
        / 1024
    )

else:

    peak_rss_kb = np.nan
    peak_rss_gib = np.nan


# =========================================================
# 10. Inventory generated output files
# =========================================================

output_files = sorted(
    glob.glob(
        PILOT_OUTPUT_PREFIX
        + "*"
    )
)


assert len(
    output_files
) > 0, (
    "unitig-caller completed but no output files "
    "were found."
)


output_inventory = []


for path in output_files:

    size_bytes = (
        os.path.getsize(
            path
        )
        if os.path.isfile(
            path
        )
        else 0
    )


    output_inventory.append(
        {
            "File":
                os.path.basename(
                    path
                ),

            "Path":
                path,

            "Size Bytes":
                size_bytes,

            "Size MiB":
                size_bytes
                / (1024 ** 2),

            "Size GiB":
                size_bytes
                / (1024 ** 3)
        }
    )


df_output_inventory = pd.DataFrame(
    output_inventory
)


total_output_bytes = int(
    df_output_inventory[
        "Size Bytes"
    ]
    .sum()
)


total_output_gib = (
    total_output_bytes
    / (1024 ** 3)
)


# =========================================================
# 11. Locate pyseer-format unitig calls
# =========================================================

pyseer_candidates = [
    path
    for path in output_files
    if "pyseer" in os.path.basename(
        path
    ).lower()
]


if not pyseer_candidates:

    # Defensive fallback in case naming changes
    pyseer_candidates = [
        path
        for path in output_files
        if path.endswith(
            ".pyseer"
        )
        or path.endswith(
            ".pyseer.gz"
        )
    ]


assert len(
    pyseer_candidates
) >= 1, (
    "Could not identify the pyseer unitig-call output."
)


# Prefer the largest matching file if multiple exist
PYSEER_OUTPUT_PATH = max(
    pyseer_candidates,
    key=os.path.getsize
)


print()
print(
    "Detected pyseer output:"
)

print(
    PYSEER_OUTPUT_PATH
)


# =========================================================
# 12. Stream-parse unitig output
# =========================================================

def open_text_auto(
    path
):

    if path.endswith(
        ".gz"
    ):

        return gzip.open(
            path,
            "rt"
        )

    return open(
        path,
        "r"
    )


n_unitigs = 0

unitig_length_sum = 0

unitig_length_min = None

unitig_length_max = 0


# prevalence can only be 0..64 in this pilot
prevalence_histogram = Counter()


observed_sample_names = set()


parse_start = time.time()


with open_text_auto(
    PYSEER_OUTPUT_PATH
) as handle:

    for line in handle:

        line = line.strip()


        if not line:
            continue


        # Expected pyseer format:
        # UNITIG_SEQUENCE | genome1:1 genome2:1 ...
        if "|" not in line:

            continue


        sequence_part, calls_part = (
            line.split(
                "|",
                1
            )
        )


        unitig_sequence = (
            sequence_part.strip()
        )


        call_tokens = (
            calls_part.strip().split()
            if calls_part.strip()
            else []
        )


        prevalence = len(
            call_tokens
        )


        prevalence_histogram[
            prevalence
        ] += 1


        n_unitigs += 1


        seq_len = len(
            unitig_sequence
        )


        unitig_length_sum += (
            seq_len
        )


        if (
            unitig_length_min is None
            or seq_len
            < unitig_length_min
        ):

            unitig_length_min = (
                seq_len
            )


        unitig_length_max = max(
            unitig_length_max,
            seq_len
        )


        # Collect sample names for colour sanity check.
        for token in call_tokens:

            sample_name = (
                token.rsplit(
                    ":",
                    1
                )[0]
            )

            observed_sample_names.add(
                sample_name
            )


parse_elapsed = (
    time.time()
    - parse_start
)


assert n_unitigs > 0


mean_unitig_length = (
    unitig_length_sum
    / n_unitigs
)


# =========================================================
# 13. Unitig prevalence summary
# =========================================================

df_prevalence_spectrum = pd.DataFrame(
    [
        {
            "Genomes Present":
                prevalence,

            "Unitigs":
                count,

            "Prevalence":
                prevalence
                / PILOT_N
        }

        for prevalence, count
        in sorted(
            prevalence_histogram.items()
        )
    ]
)


singleton_unitigs = int(
    prevalence_histogram.get(
        1,
        0
    )
)


core_unitigs = int(
    prevalence_histogram.get(
        PILOT_N,
        0
    )
)


variable_unitigs = int(
    n_unitigs
    - core_unitigs
)


# Minor-state >=5 equivalent for pilot diagnostic only.
pilot_msc5_unitigs = int(
    sum(
        count
        for prevalence, count
        in prevalence_histogram.items()
        if min(
            prevalence,
            PILOT_N - prevalence
        ) >= 5
    )
)


# =========================================================
# 14. Post-run disk audit
# =========================================================

disk_after = shutil.disk_usage(
    "/content"
)


disk_free_after_gib = (
    disk_after.free
    / (1024 ** 3)
)


local_disk_consumed_gib = (
    disk_free_before_gib
    - disk_free_after_gib
)


# =========================================================
# 15. Benchmark summary
# =========================================================

df_pilot_benchmark = pd.DataFrame(
    [
        {
            "Pilot Genomes":
                PILOT_N,

            "k-mer Size":
                UNITIG_K,

            "Threads":
                UNITIG_THREADS,

            "Compressed FASTA MiB":
                df_pilot_04[
                    "FASTA Size MiB"
                ].sum(),

            "Uncompressed FASTA GiB":
                staged_gib,

            "Staging Seconds":
                staging_elapsed,

            "Unitig Caller Wall Seconds":
                wall_seconds,

            "Unitig Caller Wall Minutes":
                wall_seconds / 60,

            "Peak RSS GiB":
                peak_rss_gib,

            "Observed Unitigs":
                n_unitigs,

            "Singleton Unitigs":
                singleton_unitigs,

            "Core Unitigs":
                core_unitigs,

            "Variable Unitigs":
                variable_unitigs,

            "Pilot Unitigs with Minor-State Count >=5":
                pilot_msc5_unitigs,

            "Minimum Unitig Length":
                unitig_length_min,

            "Mean Unitig Length":
                mean_unitig_length,

            "Maximum Unitig Length":
                unitig_length_max,

            "Output GiB":
                total_output_gib,

            "Local Disk Consumed GiB":
                local_disk_consumed_gib,

            "Parse Seconds":
                parse_elapsed,

            "Phenotype Used":
                False
        }
    ]
)


# =========================================================
# 16. Save persistent benchmark records
# =========================================================

df_pilot_benchmark.to_csv(
    PILOT_BENCHMARK_PATH,
    index=False
)


df_prevalence_spectrum.to_csv(
    PILOT_PREVALENCE_PATH,
    index=False
)


df_output_inventory.to_csv(
    PILOT_OUTPUT_INVENTORY_PATH,
    index=False
)


# =========================================================
# 17. Final report
# =========================================================

print()
print("=" * 70)
print("UNITIG PILOT BENCHMARK")
print("=" * 70)

print(
    f"Pilot genomes                 : "
    f"{PILOT_N:,}"
)

print(
    f"k-mer size                    : "
    f"{UNITIG_K}"
)

print(
    f"Threads                       : "
    f"{UNITIG_THREADS}"
)

print(
    f"Wall runtime                  : "
    f"{wall_seconds / 60:.2f} min"
)

print(
    f"Peak RSS                      : "
    f"{peak_rss_gib:.2f} GiB"
)

print(
    f"Observed unitigs              : "
    f"{n_unitigs:,}"
)

print(
    f"Singleton unitigs             : "
    f"{singleton_unitigs:,}"
)

print(
    f"Core unitigs (64/64)          : "
    f"{core_unitigs:,}"
)

print(
    f"Variable unitigs              : "
    f"{variable_unitigs:,}"
)

print(
    f"Pilot MSC>=5 unitigs          : "
    f"{pilot_msc5_unitigs:,}"
)

print()

print(
    f"Unitig length min/mean/max    : "
    f"{unitig_length_min:,} / "
    f"{mean_unitig_length:.1f} / "
    f"{unitig_length_max:,} bp"
)

print()

print(
    f"Generated output size         : "
    f"{total_output_gib:.3f} GiB"
)

print(
    f"Total local disk consumed     : "
    f"{local_disk_consumed_gib:.3f} GiB"
)

print(
    f"Local disk remaining          : "
    f"{disk_free_after_gib:.2f} GiB"
)

print()

print(
    "Output file inventory:"
)

display(
    df_output_inventory
    .sort_values(
        "Size Bytes",
        ascending=False
    )
)


print()
print(
    "Unitig prevalence spectrum:"
)

display(
    df_prevalence_spectrum
)


print()
print(
    "Benchmark saved to:"
)

print(
    PILOT_BENCHMARK_PATH
)

print()

print(
    "PASS: The 64-genome unitig pilot completed. "
    "Full-cohort feasibility can now be evaluated "
    "from measured runtime, memory, dimensionality, "
    "and storage requirements."
)

64-GENOME UNITIG PILOT
Pilot genomes        : 64
k-mer size           : 31
Threads              : 2
Local free disk      : 85.45 GiB
Staged 16/64 assemblies
Staged 32/64 assemblies
Staged 48/64 assemblies
Staged 64/64 assemblies

Uncompressed pilot FASTA: 0.334 GiB
Staging runtime          : 8.6 s

RUNNING UNITIG-CALLER
/usr/bin/time -v /content/unitig_env/bin/unitig-caller --call --refs /content/sequence_variation_work/unitig_pilot64/refs.txt --out /content/sequence_variation_work/unitig_pilot64/output/pilot64_k31 --kmer 31 --threads 2 --pyseer --write-graph

Building a DBG and calling unitigs within...
Building coloured compacted DBG...
Calling unitigs within population...
	Command being timed: "/content/unitig_env/bin/unitig-caller --call --refs /content/sequence_variation_work/unitig_pilot64/refs.txt --out /content/sequence_variation_work/unitig_pilot64/output/pilot64_k31 --kmer 31 --threads 2 --pyseer --write-graph"
	User time (seconds): 575.22
	System time (seconds): 5.50
	Percen

,File,Path,Size Bytes,Size MiB,Size GiB
0,pilot64_k31.pyseer,/content/sequence_variation_work/unitig_pilot6...,357579890,341.014757,0.333022



Unitig prevalence spectrum:


,Genomes Present,Unitigs,Prevalence
0,1,226629,0.015625
1,2,123351,0.031250
2,3,55577,0.046875
3,4,27441,0.062500
4,5,18717,0.078125
...,...,...,...
59,60,49515,0.937500
60,61,42628,0.953125
61,62,27866,0.968750
62,63,20493,0.984375



Benchmark saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/sequence_variation/unitig_pilot64_benchmark.csv

PASS: The 64-genome unitig pilot completed. Full-cohort feasibility can now be evaluated from measured runtime, memory, dimensionality, and storage requirements.


# Cell 5 — Intermediate 256-Genome Unitig Scaling Benchmark

In [ ]:
import os
import re
import gzip
import glob
import time
import shutil
import subprocess

from collections import Counter

import numpy as np
import pandas as pd


# =========================================================
# 1. Parameters
# =========================================================

BENCHMARK_N = 256

UNITIG_K = 31

UNITIG_THREADS = 2


BENCHMARK_ROOT = os.path.join(
    SEQUENCE_TMP_DIR,
    "unitig_pilot256"
)

BENCHMARK_FASTA_DIR = os.path.join(
    BENCHMARK_ROOT,
    "assemblies"
)

BENCHMARK_OUTPUT_DIR = os.path.join(
    BENCHMARK_ROOT,
    "output"
)

BENCHMARK_REFS_PATH = os.path.join(
    BENCHMARK_ROOT,
    "refs.txt"
)

BENCHMARK_OUTPUT_PREFIX = os.path.join(
    BENCHMARK_OUTPUT_DIR,
    "pilot256_k31"
)


BENCHMARK_MANIFEST_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_pilot256_manifest.csv"
)

BENCHMARK_SUMMARY_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_pilot256_benchmark.csv"
)

BENCHMARK_PREVALENCE_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_pilot256_prevalence_spectrum.csv"
)

BENCHMARK_LOG_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_pilot256_run.log"
)


# =========================================================
# 2. Build phenotype-independent 256-genome pilot
# =========================================================

df_source_256 = (
    df_sequence_input[
        [
            "Genome ID",
            "FASTA Path"
        ]
    ]
    .copy()
)


df_source_256[
    "FASTA Size Bytes"
] = (
    df_source_256[
        "FASTA Path"
    ]
    .astype(str)
    .apply(
        os.path.getsize
    )
)


df_source_256[
    "FASTA Size MiB"
] = (
    df_source_256[
        "FASTA Size Bytes"
    ]
    / (1024 ** 2)
)


df_source_256 = (
    df_source_256
    .sort_values(
        [
            "FASTA Size Bytes",
            "Genome ID"
        ]
    )
    .reset_index(
        drop=True
    )
)


pilot_positions = np.linspace(
    0,
    len(
        df_source_256
    ) - 1,
    BENCHMARK_N,
    dtype=int
)


assert (
    len(
        np.unique(
            pilot_positions
        )
    )
    == BENCHMARK_N
)


df_pilot_256 = (
    df_source_256
    .iloc[
        pilot_positions
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


df_pilot_256[
    "Pilot Index"
] = np.arange(
    1,
    BENCHMARK_N + 1
)


df_pilot_256 = (
    df_pilot_256[
        [
            "Pilot Index",
            "Genome ID",
            "FASTA Path",
            "FASTA Size Bytes",
            "FASTA Size MiB"
        ]
    ]
)


assert len(
    df_pilot_256
) == BENCHMARK_N


assert (
    df_pilot_256[
        "Genome ID"
    ]
    .nunique()
    == BENCHMARK_N
)


assert (
    "Resistant Phenotype"
    not in df_pilot_256.columns
)


df_pilot_256.to_csv(
    BENCHMARK_MANIFEST_PATH,
    index=False
)


# =========================================================
# 3. Disk audit
# =========================================================

disk_before = shutil.disk_usage(
    "/content"
)


disk_free_before_gib = (
    disk_before.free
    / (1024 ** 3)
)


print("=" * 70)
print("256-GENOME UNITIG SCALING BENCHMARK")
print("=" * 70)

print(
    f"Pilot genomes        : "
    f"{BENCHMARK_N}"
)

print(
    f"k-mer size           : "
    f"{UNITIG_K}"
)

print(
    f"Threads              : "
    f"{UNITIG_THREADS}"
)

print(
    f"Local free disk      : "
    f"{disk_free_before_gib:.2f} GiB"
)


assert disk_free_before_gib >= 20, (
    "Insufficient local disk for 256-genome benchmark."
)


# =========================================================
# 4. Clean previous local benchmark data
# =========================================================

if os.path.exists(
    BENCHMARK_ROOT
):

    shutil.rmtree(
        BENCHMARK_ROOT
    )


os.makedirs(
    BENCHMARK_FASTA_DIR,
    exist_ok=True
)

os.makedirs(
    BENCHMARK_OUTPUT_DIR,
    exist_ok=True
)


# =========================================================
# 5. Stage 256 assemblies
# =========================================================

staging_records = []

staging_start = time.time()


for i, row in df_pilot_256.iterrows():

    genome_id = row[
        "Genome ID"
    ]

    source_path = str(
        row[
            "FASTA Path"
        ]
    )


    destination_path = os.path.join(
        BENCHMARK_FASTA_DIR,
        f"{genome_id}.fasta"
    )


    if source_path.endswith(
        ".gz"
    ):

        with gzip.open(
            source_path,
            "rb"
        ) as src, open(
            destination_path,
            "wb"
        ) as dst:

            shutil.copyfileobj(
                src,
                dst,
                length=1024 * 1024
            )

    else:

        shutil.copyfile(
            source_path,
            destination_path
        )


    size_bytes = os.path.getsize(
        destination_path
    )


    assert size_bytes > 0


    staging_records.append(
        {
            "Genome ID":
                genome_id,

            "Local FASTA":
                destination_path,

            "Uncompressed Bytes":
                size_bytes
        }
    )


    if (
        (i + 1) % 64 == 0
        or (i + 1) == BENCHMARK_N
    ):

        print(
            f"Staged "
            f"{i + 1}/{BENCHMARK_N} assemblies"
        )


df_stage_256 = pd.DataFrame(
    staging_records
)


staging_seconds = (
    time.time()
    - staging_start
)


staged_gib = (
    df_stage_256[
        "Uncompressed Bytes"
    ]
    .sum()
    / (1024 ** 3)
)


print()

print(
    f"Uncompressed pilot FASTA: "
    f"{staged_gib:.3f} GiB"
)

print(
    f"Staging runtime          : "
    f"{staging_seconds:.1f} s"
)


# =========================================================
# 6. Build refs.txt
# =========================================================

with open(
    BENCHMARK_REFS_PATH,
    "w"
) as handle:

    for path in df_stage_256[
        "Local FASTA"
    ]:

        handle.write(
            os.path.abspath(
                path
            )
            + "\n"
        )


# =========================================================
# 7. Run unitig-caller
# =========================================================

command = [
    "/usr/bin/time",
    "-v",
    UNITIG_CALLER_BIN,
    "--call",
    "--refs",
    BENCHMARK_REFS_PATH,
    "--out",
    BENCHMARK_OUTPUT_PREFIX,
    "--kmer",
    str(
        UNITIG_K
    ),
    "--threads",
    str(
        UNITIG_THREADS
    ),
    "--pyseer"
]


print()
print("=" * 70)
print("RUNNING 256-GENOME UNITIG BENCHMARK")
print("=" * 70)

print(
    " ".join(
        command
    )
)

print()


run_start = time.time()


result = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)


wall_seconds = (
    time.time()
    - run_start
)


with open(
    BENCHMARK_LOG_PATH,
    "w"
) as handle:

    handle.write(
        result.stdout
    )


print(
    result.stdout
)


if result.returncode != 0:

    raise RuntimeError(
        "256-genome unitig benchmark failed. "
        f"See log: {BENCHMARK_LOG_PATH}"
    )


# =========================================================
# 8. Peak RAM
# =========================================================

rss_match = re.search(
    r"Maximum resident set size \(kbytes\):\s*(\d+)",
    result.stdout
)


if rss_match:

    peak_rss_kb = int(
        rss_match.group(1)
    )

    peak_rss_gib = (
        peak_rss_kb
        / 1024
        / 1024
    )

else:

    peak_rss_gib = np.nan


# =========================================================
# 9. Detect pyseer output
# =========================================================

output_files = sorted(
    glob.glob(
        BENCHMARK_OUTPUT_PREFIX
        + "*"
    )
)


pyseer_candidates = [
    path
    for path in output_files
    if "pyseer"
    in os.path.basename(
        path
    ).lower()
]


assert len(
    pyseer_candidates
) >= 1


PYSEER_256_PATH = max(
    pyseer_candidates,
    key=os.path.getsize
)


output_size_gib = (
    os.path.getsize(
        PYSEER_256_PATH
    )
    / (1024 ** 3)
)


print()
print(
    "Detected pyseer output:"
)

print(
    PYSEER_256_PATH
)


# =========================================================
# 10. Stream prevalence counts
# =========================================================

def open_text_auto(
    path
):

    if path.endswith(
        ".gz"
    ):

        return gzip.open(
            path,
            "rt"
        )

    return open(
        path,
        "r"
    )


prevalence_histogram = Counter()

n_unitigs = 0

unitig_length_sum = 0

unitig_length_min = None

unitig_length_max = 0


parse_start = time.time()


with open_text_auto(
    PYSEER_256_PATH
) as handle:

    for line in handle:

        line = line.strip()


        if not line:
            continue


        if "|" not in line:
            continue


        sequence_part, calls_part = (
            line.split(
                "|",
                1
            )
        )


        sequence_part = (
            sequence_part.strip()
        )


        calls = (
            calls_part.strip().split()
            if calls_part.strip()
            else []
        )


        prevalence = len(
            calls
        )


        prevalence_histogram[
            prevalence
        ] += 1


        n_unitigs += 1


        seq_len = len(
            sequence_part
        )


        unitig_length_sum += (
            seq_len
        )


        if (
            unitig_length_min is None
            or seq_len
            < unitig_length_min
        ):

            unitig_length_min = (
                seq_len
            )


        unitig_length_max = max(
            unitig_length_max,
            seq_len
        )


parse_seconds = (
    time.time()
    - parse_start
)


mean_unitig_length = (
    unitig_length_sum
    / n_unitigs
)


# =========================================================
# 11. Prevalence diagnostics
# =========================================================

singleton_unitigs = int(
    prevalence_histogram.get(
        1,
        0
    )
)


core_unitigs = int(
    prevalence_histogram.get(
        BENCHMARK_N,
        0
    )
)


variable_unitigs = (
    n_unitigs
    - core_unitigs
)


msc5_unitigs = int(
    sum(
        count
        for prevalence, count
        in prevalence_histogram.items()
        if min(
            prevalence,
            BENCHMARK_N - prevalence
        ) >= 5
    )
)


df_prevalence_256 = pd.DataFrame(
    [
        {
            "Genomes Present":
                prevalence,

            "Unitigs":
                count,

            "Prevalence":
                prevalence
                / BENCHMARK_N
        }

        for prevalence, count
        in sorted(
            prevalence_histogram.items()
        )
    ]
)


# =========================================================
# 12. Save benchmark summary
# =========================================================

df_benchmark_256 = pd.DataFrame(
    [
        {
            "Pilot Genomes":
                BENCHMARK_N,

            "k-mer Size":
                UNITIG_K,

            "Threads":
                UNITIG_THREADS,

            "Uncompressed FASTA GiB":
                staged_gib,

            "Wall Seconds":
                wall_seconds,

            "Wall Minutes":
                wall_seconds / 60,

            "Peak RSS GiB":
                peak_rss_gib,

            "Observed Unitigs":
                n_unitigs,

            "Singleton Unitigs":
                singleton_unitigs,

            "Core Unitigs":
                core_unitigs,

            "Variable Unitigs":
                variable_unitigs,

            "MSC>=5 Unitigs":
                msc5_unitigs,

            "Minimum Unitig Length":
                unitig_length_min,

            "Mean Unitig Length":
                mean_unitig_length,

            "Maximum Unitig Length":
                unitig_length_max,

            "Pyseer Output GiB":
                output_size_gib,

            "Phenotype Used":
                False
        }
    ]
)


df_benchmark_256.to_csv(
    BENCHMARK_SUMMARY_PATH,
    index=False
)


df_prevalence_256.to_csv(
    BENCHMARK_PREVALENCE_PATH,
    index=False
)


# =========================================================
# 13. Compare against 64-genome pilot
# =========================================================

df_benchmark_64 = pd.read_csv(
    PILOT_BENCHMARK_PATH
)


row64 = df_benchmark_64.iloc[
    0
]


runtime_ratio = (
    wall_seconds
    / float(
        row64[
            "Unitig Caller Wall Seconds"
        ]
    )
)


ram_ratio = (
    peak_rss_gib
    / float(
        row64[
            "Peak RSS GiB"
        ]
    )
)


unitig_ratio = (
    n_unitigs
    / int(
        row64[
            "Observed Unitigs"
        ]
    )
)


output_ratio = (
    output_size_gib
    / float(
        row64[
            "Output GiB"
        ]
    )
)


# =========================================================
# 14. Final report
# =========================================================

print()
print("=" * 70)
print("256-GENOME UNITIG SCALING RESULT")
print("=" * 70)

print(
    f"Pilot genomes              : "
    f"{BENCHMARK_N:,}"
)

print(
    f"Wall runtime               : "
    f"{wall_seconds / 60:.2f} min"
)

print(
    f"Peak RSS                   : "
    f"{peak_rss_gib:.2f} GiB"
)

print(
    f"Observed unitigs           : "
    f"{n_unitigs:,}"
)

print(
    f"Singleton unitigs          : "
    f"{singleton_unitigs:,}"
)

print(
    f"Core unitigs (256/256)     : "
    f"{core_unitigs:,}"
)

print(
    f"Variable unitigs           : "
    f"{variable_unitigs:,}"
)

print(
    f"MSC>=5 unitigs             : "
    f"{msc5_unitigs:,}"
)

print(
    f"Pyseer output              : "
    f"{output_size_gib:.3f} GiB"
)

print()

print(
    "Scaling relative to 64-genome pilot:"
)

print(
    f"Runtime ratio              : "
    f"{runtime_ratio:.2f}x"
)

print(
    f"Peak RAM ratio             : "
    f"{ram_ratio:.2f}x"
)

print(
    f"Unitig-count ratio         : "
    f"{unitig_ratio:.2f}x"
)

print(
    f"Output-size ratio          : "
    f"{output_ratio:.2f}x"
)

print()

print(
    f"Unitig length min/mean/max : "
    f"{unitig_length_min:,} / "
    f"{mean_unitig_length:.1f} / "
    f"{unitig_length_max:,} bp"
)

print()

print(
    "Benchmark saved to:"
)

print(
    BENCHMARK_SUMMARY_PATH
)

print()

print(
    "PASS: The 256-genome scaling benchmark completed. "
    "Full-cohort feasibility can now be assessed using "
    "two measured cohort sizes rather than a single pilot."
)

256-GENOME UNITIG SCALING BENCHMARK
Pilot genomes        : 256
k-mer size           : 31
Threads              : 2
Local free disk      : 84.95 GiB
Staged 64/256 assemblies
Staged 128/256 assemblies
Staged 192/256 assemblies
Staged 256/256 assemblies

Uncompressed pilot FASTA: 1.343 GiB
Staging runtime          : 201.3 s

RUNNING 256-GENOME UNITIG BENCHMARK
/usr/bin/time -v /content/unitig_env/bin/unitig-caller --call --refs /content/sequence_variation_work/unitig_pilot256/refs.txt --out /content/sequence_variation_work/unitig_pilot256/output/pilot256_k31 --kmer 31 --threads 2 --pyseer

Building a DBG and calling unitigs within...
Building coloured compacted DBG...
Calling unitigs within population...
	Command being timed: "/content/unitig_env/bin/unitig-caller --call --refs /content/sequence_variation_work/unitig_pilot256/refs.txt --out /content/sequence_variation_work/unitig_pilot256/output/pilot256_k31 --kmer 31 --threads 2 --pyseer"
	User time (seconds): 2213.92
	System time (second

# Cell 6 — Evaluate Full-Cohort Unitig Feasibility

In [ ]:
import os
import math
import shutil
import pandas as pd


# =========================================================
# 1. Inputs
# =========================================================

FULL_N = 4227

BENCHMARK_64_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_pilot64_benchmark.csv"
)

BENCHMARK_256_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_pilot256_benchmark.csv"
)

FEASIBILITY_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_full_cohort_feasibility.csv"
)


df64 = pd.read_csv(
    BENCHMARK_64_PATH
)

df256 = pd.read_csv(
    BENCHMARK_256_PATH
)


row64 = df64.iloc[0]
row256 = df256.iloc[0]


# =========================================================
# 2. Scaling helper
#
# Assume metric ~ N^p, estimate p from 64 -> 256.
# =========================================================

def scaling_exponent(
    value64,
    value256
):

    return (
        math.log(
            value256 / value64
        )
        /
        math.log(
            256 / 64
        )
    )


def project_metric(
    value256,
    exponent
):

    return (
        value256
        *
        (FULL_N / 256) ** exponent
    )


# =========================================================
# 3. Runtime projection
# =========================================================

runtime_p = scaling_exponent(
    float(
        row64[
            "Unitig Caller Wall Minutes"
        ]
    ),
    float(
        row256[
            "Wall Minutes"
        ]
    )
)


projected_runtime_min = project_metric(
    float(
        row256[
            "Wall Minutes"
        ]
    ),
    runtime_p
)


projected_runtime_hr = (
    projected_runtime_min
    / 60
)


# =========================================================
# 4. RAM projection
# =========================================================

ram_p = scaling_exponent(
    float(
        row64[
            "Peak RSS GiB"
        ]
    ),
    float(
        row256[
            "Peak RSS GiB"
        ]
    )
)


projected_ram_gib = project_metric(
    float(
        row256[
            "Peak RSS GiB"
        ]
    ),
    ram_p
)


# =========================================================
# 5. Unitig-count projection
# =========================================================

unitig_p = scaling_exponent(
    float(
        row64[
            "Observed Unitigs"
        ]
    ),
    float(
        row256[
            "Observed Unitigs"
        ]
    )
)


projected_unitigs = project_metric(
    float(
        row256[
            "Observed Unitigs"
        ]
    ),
    unitig_p
)


# =========================================================
# 6. Pyseer-output projection
# =========================================================

output_p = scaling_exponent(
    float(
        row64[
            "Output GiB"
        ]
    ),
    float(
        row256[
            "Pyseer Output GiB"
        ]
    )
)


projected_pyseer_gib = project_metric(
    float(
        row256[
            "Pyseer Output GiB"
        ]
    ),
    output_p
)


# =========================================================
# 7. Uncompressed FASTA projection
# =========================================================

projected_fasta_gib = (
    float(
        row256[
            "Uncompressed FASTA GiB"
        ]
    )
    *
    FULL_N
    / 256
)


# =========================================================
# 8. Local disk status
# =========================================================

disk = shutil.disk_usage(
    "/content"
)


local_free_gib = (
    disk.free
    / (1024 ** 3)
)


projected_primary_footprint = (
    projected_fasta_gib
    +
    projected_pyseer_gib
)


footprint_fraction = (
    projected_primary_footprint
    /
    local_free_gib
)


remaining_after_primary_gib = (
    local_free_gib
    -
    projected_primary_footprint
)


# =========================================================
# 9. Conservative direct-run decision
#
# Require projected FASTA + main output to use <=70%
# of currently free local disk.
# =========================================================

DISK_SAFETY_LIMIT = 0.70


direct_run_approved = bool(
    footprint_fraction
    <= DISK_SAFETY_LIMIT
)


decision = (
    "Direct full run approved"
    if direct_run_approved
    else
    (
        "Direct full run NOT approved; "
        "use a storage-safe full-cohort execution strategy."
    )
)


# =========================================================
# 10. Save feasibility record
# =========================================================

df_feasibility = pd.DataFrame(
    [
        {
            "Full Cohort Genomes":
                FULL_N,

            "Runtime Scaling Exponent":
                runtime_p,

            "Projected Runtime Hours":
                projected_runtime_hr,

            "RAM Scaling Exponent":
                ram_p,

            "Projected Peak RSS GiB":
                projected_ram_gib,

            "Unitig Scaling Exponent":
                unitig_p,

            "Projected Unitigs":
                projected_unitigs,

            "Output Scaling Exponent":
                output_p,

            "Projected Pyseer GiB":
                projected_pyseer_gib,

            "Projected Uncompressed FASTA GiB":
                projected_fasta_gib,

            "Projected Primary Local Footprint GiB":
                projected_primary_footprint,

            "Current Local Free Disk GiB":
                local_free_gib,

            "Projected Remaining Disk GiB":
                remaining_after_primary_gib,

            "Projected Disk Fraction":
                footprint_fraction,

            "Disk Safety Limit":
                DISK_SAFETY_LIMIT,

            "Direct Full Run Approved":
                direct_run_approved,

            "Phenotype Used":
                False,

            "Decision":
                decision
        }
    ]
)


df_feasibility.to_csv(
    FEASIBILITY_PATH,
    index=False
)


# =========================================================
# 11. Report
# =========================================================

print("=" * 70)
print("FULL-COHORT UNITIG FEASIBILITY")
print("=" * 70)

print(
    f"Full cohort                     : "
    f"{FULL_N:,}"
)

print()

print(
    f"Projected runtime               : "
    f"{projected_runtime_hr:.2f} h"
)

print(
    f"Projected peak RAM              : "
    f"{projected_ram_gib:.2f} GiB"
)

print(
    f"Projected unitigs               : "
    f"{projected_unitigs:,.0f}"
)

print()

print(
    f"Projected uncompressed FASTA    : "
    f"{projected_fasta_gib:.2f} GiB"
)

print(
    f"Projected pyseer output         : "
    f"{projected_pyseer_gib:.2f} GiB"
)

print(
    f"Projected primary footprint     : "
    f"{projected_primary_footprint:.2f} GiB"
)

print(
    f"Current local free disk         : "
    f"{local_free_gib:.2f} GiB"
)

print(
    f"Projected disk utilization      : "
    f"{100 * footprint_fraction:.1f}%"
)

print(
    f"Projected remaining disk        : "
    f"{remaining_after_primary_gib:.2f} GiB"
)

print()

print(
    f"Direct full run approved        : "
    f"{direct_run_approved}"
)

print()

print(
    "Decision:"
)

print(
    decision
)

print()

print(
    "Feasibility record saved to:"
)

print(
    FEASIBILITY_PATH
)

print()

print(
    "PASS: Full-cohort feasibility was evaluated "
    "using measured 64- and 256-genome benchmarks."
)

FULL-COHORT UNITIG FEASIBILITY
Full cohort                     : 4,227

Projected runtime               : 6.38 h
Projected peak RAM              : 2.20 GiB
Projected unitigs               : 6,581,083

Projected uncompressed FASTA    : 22.18 GiB
Projected pyseer output         : 48.03 GiB
Projected primary footprint     : 70.21 GiB
Current local free disk         : 81.41 GiB
Projected disk utilization      : 86.2%
Projected remaining disk        : 11.21 GiB

Direct full run approved        : False

Decision:
Direct full run NOT approved; use a storage-safe full-cohort execution strategy.

Feasibility record saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/sequence_variation/unitig_full_cohort_feasibility.csv

PASS: Full-cohort feasibility was evaluated using measured 64- and 256-genome benchmarks.


# Cell 7 — Validate the Storage-Safe Unitig Execution Strategy

In [ ]:
import os
import re
import gzip
import time
import shutil
import subprocess

import pandas as pd


# =========================================================
# 1. Paths and parameters
# =========================================================

SAFE_TEST_N = 64
UNITIG_K = 31
UNITIG_THREADS = 2

SAFE_ROOT = os.path.join(
    SEQUENCE_TMP_DIR,
    "unitig_storage_safe_test64"
)

SAFE_OUTPUT_DIR = os.path.join(
    SAFE_ROOT,
    "output"
)

SAFE_REFS_PATH = os.path.join(
    SAFE_ROOT,
    "refs_gz.txt"
)

SAFE_OUTPUT_PREFIX = os.path.join(
    SAFE_OUTPUT_DIR,
    "pilot64_k31_safe"
)

SAFE_FIFO_PATH = (
    SAFE_OUTPUT_PREFIX
    + ".pyseer"
)

SAFE_GZIP_PATH = (
    SAFE_OUTPUT_PREFIX
    + ".pyseer.gz"
)

SAFE_LOG_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_storage_safe_test64.log"
)

SAFE_SUMMARY_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_storage_safe_test64_summary.csv"
)


# =========================================================
# 2. Reload original 64-genome pilot manifest
# =========================================================

df_safe64 = pd.read_csv(
    UNITIG_PILOT_MANIFEST_PATH,
    dtype={
        "Genome ID": str
    }
)


assert len(
    df_safe64
) == SAFE_TEST_N


assert (
    df_safe64[
        "Genome ID"
    ]
    .nunique()
    == SAFE_TEST_N
)


assert (
    df_safe64[
        "FASTA Path"
    ]
    .apply(
        os.path.exists
    )
    .all()
)


# No phenotype used
assert (
    "Resistant Phenotype"
    not in df_safe64.columns
)


# =========================================================
# 3. Confirm compressed-input status
# =========================================================

compressed_count = int(
    df_safe64[
        "FASTA Path"
    ]
    .astype(str)
    .str.endswith(
        ".gz"
    )
    .sum()
)


assert compressed_count == SAFE_TEST_N, (
    "Not all pilot FASTA files are gzip-compressed."
)


compressed_input_gib = (
    df_safe64[
        "FASTA Path"
    ]
    .astype(str)
    .apply(
        os.path.getsize
    )
    .sum()
    / (1024 ** 3)
)


# =========================================================
# 4. Clean test workspace
# =========================================================

if os.path.exists(
    SAFE_ROOT
):

    shutil.rmtree(
        SAFE_ROOT
    )


os.makedirs(
    SAFE_OUTPUT_DIR,
    exist_ok=True
)


# =========================================================
# 5. Create refs.txt pointing DIRECTLY to .fna.gz
# =========================================================

with open(
    SAFE_REFS_PATH,
    "w"
) as handle:

    for path in df_safe64[
        "FASTA Path"
    ]:

        handle.write(
            os.path.abspath(
                str(
                    path
                )
            )
            + "\n"
        )


with open(
    SAFE_REFS_PATH,
    "r"
) as handle:

    refs = [
        line.strip()
        for line in handle
        if line.strip()
    ]


assert len(
    refs
) == SAFE_TEST_N


assert all(
    path.endswith(
        ".gz"
    )
    for path in refs
)


# =========================================================
# 6. Create FIFO for streaming pyseer output
# =========================================================

if os.path.exists(
    SAFE_FIFO_PATH
):

    os.remove(
        SAFE_FIFO_PATH
    )


if os.path.exists(
    SAFE_GZIP_PATH
):

    os.remove(
        SAFE_GZIP_PATH
    )


os.mkfifo(
    SAFE_FIFO_PATH
)


assert os.path.exists(
    SAFE_FIFO_PATH
)


# =========================================================
# 7. Start gzip reader BEFORE unitig-caller
#
# gzip -1 minimizes CPU overhead while still reducing
# the huge repetitive pyseer text output.
# =========================================================

gzip_command = (
    f"gzip -1 -c < "
    f"'{SAFE_FIFO_PATH}' "
    f"> '{SAFE_GZIP_PATH}'"
)


gzip_process = subprocess.Popen(
    [
        "bash",
        "-c",
        gzip_command
    ]
)


# Give the reader a moment to attach to the FIFO
time.sleep(
    1
)


# =========================================================
# 8. Run unitig-caller directly on compressed FASTA
# =========================================================

command = [
    "/usr/bin/time",
    "-v",
    UNITIG_CALLER_BIN,
    "--call",
    "--refs",
    SAFE_REFS_PATH,
    "--out",
    SAFE_OUTPUT_PREFIX,
    "--kmer",
    str(
        UNITIG_K
    ),
    "--threads",
    str(
        UNITIG_THREADS
    ),
    "--pyseer"
]


print("=" * 70)
print("STORAGE-SAFE UNITIG VALIDATION")
print("=" * 70)

print(
    f"Pilot genomes                : "
    f"{SAFE_TEST_N}"
)

print(
    f"Input format                 : "
    f"compressed .fna.gz"
)

print(
    f"Compressed input size        : "
    f"{compressed_input_gib:.3f} GiB"
)

print(
    "Pyseer handling              : "
    "FIFO -> gzip -1"
)

print()

print(
    "Running:"
)

print(
    " ".join(
        command
    )
)

print()


run_start = time.time()


result = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)


wall_seconds = (
    time.time()
    - run_start
)


# =========================================================
# 9. Wait for gzip to finish
# =========================================================

gzip_returncode = gzip_process.wait()


with open(
    SAFE_LOG_PATH,
    "w"
) as handle:

    handle.write(
        result.stdout
    )


print(
    result.stdout
)


if result.returncode != 0:

    raise RuntimeError(
        "unitig-caller storage-safe test failed. "
        f"See log: {SAFE_LOG_PATH}"
    )


if gzip_returncode != 0:

    raise RuntimeError(
        "gzip streaming process failed."
    )


# =========================================================
# 10. Remove FIFO after successful run
# =========================================================

if os.path.exists(
    SAFE_FIFO_PATH
):

    os.remove(
        SAFE_FIFO_PATH
    )


# =========================================================
# 11. Verify gzip integrity
# =========================================================

gzip_test = subprocess.run(
    [
        "gzip",
        "-t",
        SAFE_GZIP_PATH
    ]
)


assert gzip_test.returncode == 0, (
    "Compressed pyseer output failed gzip integrity check."
)


assert os.path.getsize(
    SAFE_GZIP_PATH
) > 0


# =========================================================
# 12. Parse peak RAM
# =========================================================

rss_match = re.search(
    r"Maximum resident set size \(kbytes\):\s*(\d+)",
    result.stdout
)


if rss_match:

    peak_rss_gib = (
        int(
            rss_match.group(1)
        )
        / 1024
        / 1024
    )

else:

    peak_rss_gib = float(
        "nan"
    )


# =========================================================
# 13. Count unitigs directly from compressed output
# =========================================================

safe_unitig_count = 0


with gzip.open(
    SAFE_GZIP_PATH,
    "rt"
) as handle:

    for line in handle:

        if "|" in line:

            safe_unitig_count += 1


assert safe_unitig_count > 0


# =========================================================
# 14. Compare with original 64-genome benchmark
# =========================================================

df_original64 = pd.read_csv(
    PILOT_BENCHMARK_PATH
)


original_unitig_count = int(
    df_original64.iloc[0][
        "Observed Unitigs"
    ]
)


original_output_gib = float(
    df_original64.iloc[0][
        "Output GiB"
    ]
)


compressed_output_gib = (
    os.path.getsize(
        SAFE_GZIP_PATH
    )
    / (1024 ** 3)
)


compression_ratio = (
    original_output_gib
    /
    compressed_output_gib
)


unitig_count_match = (
    safe_unitig_count
    ==
    original_unitig_count
)


# =========================================================
# 15. Local disk audit
# =========================================================

safe_root_bytes = 0


for root, dirs, files in os.walk(
    SAFE_ROOT
):

    for name in files:

        path = os.path.join(
            root,
            name
        )

        safe_root_bytes += (
            os.path.getsize(
                path
            )
        )


safe_root_gib = (
    safe_root_bytes
    / (1024 ** 3)
)


# =========================================================
# 16. Save validation summary
# =========================================================

df_safe_summary = pd.DataFrame(
    [
        {
            "Pilot Genomes":
                SAFE_TEST_N,

            "Compressed FASTA Input":
                True,

            "Input GiB":
                compressed_input_gib,

            "Streaming Compression":
                True,

            "Compression Method":
                "gzip -1 via FIFO",

            "Wall Minutes":
                wall_seconds / 60,

            "Peak RSS GiB":
                peak_rss_gib,

            "Observed Unitigs":
                safe_unitig_count,

            "Original Pilot Unitigs":
                original_unitig_count,

            "Unitig Count Match":
                unitig_count_match,

            "Original Plain Pyseer GiB":
                original_output_gib,

            "Compressed Pyseer GiB":
                compressed_output_gib,

            "Compression Ratio":
                compression_ratio,

            "Local Workspace GiB":
                safe_root_gib,

            "Phenotype Used":
                False
        }
    ]
)


df_safe_summary.to_csv(
    SAFE_SUMMARY_PATH,
    index=False
)


# =========================================================
# 17. Final report
# =========================================================

print()
print("=" * 70)
print("STORAGE-SAFE VALIDATION RESULT")
print("=" * 70)

print(
    f"Direct .fna.gz input          : PASS"
)

print(
    f"Streaming gzip output         : PASS"
)

print(
    f"Wall runtime                  : "
    f"{wall_seconds / 60:.2f} min"
)

print(
    f"Peak RSS                      : "
    f"{peak_rss_gib:.2f} GiB"
)

print()

print(
    f"Observed unitigs              : "
    f"{safe_unitig_count:,}"
)

print(
    f"Original 64-pilot unitigs     : "
    f"{original_unitig_count:,}"
)

print(
    f"Unitig count match            : "
    f"{unitig_count_match}"
)

print()

print(
    f"Original plain pyseer         : "
    f"{original_output_gib:.3f} GiB"
)

print(
    f"Compressed pyseer             : "
    f"{compressed_output_gib:.3f} GiB"
)

print(
    f"Compression ratio             : "
    f"{compression_ratio:.2f}x"
)

print(
    f"Local workspace               : "
    f"{safe_root_gib:.3f} GiB"
)

print()

print(
    "Summary saved to:"
)

print(
    SAFE_SUMMARY_PATH
)

print()

if unitig_count_match:

    print(
        "PASS: Direct compressed FASTA input and "
        "stream-compressed pyseer output reproduce "
        "the original 64-genome unitig count."
    )

else:

    print(
        "REVIEW REQUIRED: Unitig count differs from "
        "the original 64-genome pilot."
    )

STORAGE-SAFE UNITIG VALIDATION
Pilot genomes                : 64
Input format                 : compressed .fna.gz
Compressed input size        : 0.116 GiB
Pyseer handling              : FIFO -> gzip -1

Running:
/usr/bin/time -v /content/unitig_env/bin/unitig-caller --call --refs /content/sequence_variation_work/unitig_storage_safe_test64/refs_gz.txt --out /content/sequence_variation_work/unitig_storage_safe_test64/output/pilot64_k31_safe --kmer 31 --threads 2 --pyseer

Building a DBG and calling unitigs within...
Building coloured compacted DBG...
Calling unitigs within population...
	Command being timed: "/content/unitig_env/bin/unitig-caller --call --refs /content/sequence_variation_work/unitig_storage_safe_test64/refs_gz.txt --out /content/sequence_variation_work/unitig_storage_safe_test64/output/pilot64_k31_safe --kmer 31 --threads 2 --pyseer"
	User time (seconds): 587.90
	System time (seconds): 4.71
	Percent of CPU this job got: 145%
	Elapsed (wall clock) time (h:mm:ss or m:ss):

# Cell 8 — Validate a Checkpointable Bifrost Graph Workflow

In [ ]:
import os
import re
import glob
import gzip
import time
import shutil
import subprocess

from collections import Counter

import pandas as pd


# =========================================================
# 0. Basic runtime recovery checks
# =========================================================

GNU_TIME = "/usr/bin/time"

if not os.path.exists(GNU_TIME):

    print(
        "GNU time missing. Restoring..."
    )

    subprocess.run(
        [
            "apt-get",
            "update",
            "-qq"
        ],
        check=True
    )

    subprocess.run(
        [
            "apt-get",
            "install",
            "-y",
            "-qq",
            "time"
        ],
        check=True
    )


assert os.path.exists(
    GNU_TIME
)


UNITIG_ENV = (
    "/content/unitig_env"
)

UNITIG_CALLER_BIN = os.path.join(
    UNITIG_ENV,
    "bin",
    "unitig-caller"
)

BIFROST_BIN = os.path.join(
    UNITIG_ENV,
    "bin",
    "Bifrost"
)


assert os.path.exists(
    UNITIG_CALLER_BIN
), (
    "unitig-caller is missing. "
    "Rerun Cell 3 first."
)


assert os.path.exists(
    BIFROST_BIN
), (
    "Bifrost is missing. "
    "Rerun Cell 3 first."
)


# =========================================================
# 1. Parameters
# =========================================================

TEST_N = 64

UNITIG_K = 31

UNITIG_THREADS = 2


# =========================================================
# 2. Required persistent upstream files
# =========================================================

UNITIG_PILOT_MANIFEST_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_pilot_manifest.csv"
)

PILOT_PREVALENCE_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_pilot64_prevalence_spectrum.csv"
)


assert os.path.exists(
    UNITIG_PILOT_MANIFEST_PATH
)


assert os.path.exists(
    PILOT_PREVALENCE_PATH
)


# =========================================================
# 3. Persistent Drive checkpoint paths
# =========================================================

PERSISTENT_GRAPH_DIR = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "pilot64_bifrost_graph"
)


os.makedirs(
    PERSISTENT_GRAPH_DIR,
    exist_ok=True
)


PERSISTENT_GRAPH_PATH = os.path.join(
    PERSISTENT_GRAPH_DIR,
    "pilot64_k31_graph.gfa.gz"
)

PERSISTENT_COLOUR_PATH = os.path.join(
    PERSISTENT_GRAPH_DIR,
    "pilot64_k31_graph.bfg_colors"
)

PERSISTENT_BUILD_META_PATH = os.path.join(
    PERSISTENT_GRAPH_DIR,
    "pilot64_graph_checkpoint_summary.csv"
)


# =========================================================
# 4. Local working paths
# =========================================================

SPLIT_ROOT = os.path.join(
    SEQUENCE_TMP_DIR,
    "unitig_split_workflow_test64"
)

LOCAL_GRAPH_DIR = os.path.join(
    SPLIT_ROOT,
    "graph"
)

LOCAL_CALL_DIR = os.path.join(
    SPLIT_ROOT,
    "calls"
)

REFS_PATH = os.path.join(
    SPLIT_ROOT,
    "refs_gz.txt"
)


LOCAL_GRAPH_PREFIX = os.path.join(
    LOCAL_GRAPH_DIR,
    "pilot64_k31_graph"
)

LOCAL_GRAPH_PATH = (
    LOCAL_GRAPH_PREFIX
    + ".gfa.gz"
)

LOCAL_COLOUR_PATH = (
    LOCAL_GRAPH_PREFIX
    + ".bfg_colors"
)


CALL_PREFIX = os.path.join(
    LOCAL_CALL_DIR,
    "pilot64_k31_from_graph"
)

CALL_FIFO = (
    CALL_PREFIX
    + ".pyseer"
)

CALL_GZIP = (
    CALL_PREFIX
    + ".pyseer.gz"
)


SUMMARY_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_checkpointable_workflow_test64.csv"
)

GRAPH_INVENTORY_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_checkpointable_graph_inventory64.csv"
)

BUILD_LOG_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "bifrost_build_test64.log"
)

CALL_LOG_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_prebuilt_graph_call_test64.log"
)


# =========================================================
# 5. Load original 64-genome pilot
# =========================================================

df_test = pd.read_csv(
    UNITIG_PILOT_MANIFEST_PATH,
    dtype={
        "Genome ID": str
    }
)


df_test[
    "Genome ID"
] = (
    df_test[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


assert len(
    df_test
) == TEST_N


assert (
    df_test[
        "Genome ID"
    ]
    .nunique()
    == TEST_N
)


assert (
    df_test[
        "FASTA Path"
    ]
    .apply(
        os.path.exists
    )
    .all()
)


assert (
    "Resistant Phenotype"
    not in df_test.columns
)


# =========================================================
# 6. Prepare fresh local workspace
# =========================================================

if os.path.exists(
    SPLIT_ROOT
):

    shutil.rmtree(
        SPLIT_ROOT
    )


os.makedirs(
    LOCAL_GRAPH_DIR,
    exist_ok=True
)

os.makedirs(
    LOCAL_CALL_DIR,
    exist_ok=True
)


# =========================================================
# 7. refs.txt uses original compressed FASTA files
# =========================================================

with open(
    REFS_PATH,
    "w"
) as handle:

    for path in df_test[
        "FASTA Path"
    ]:

        path = os.path.abspath(
            str(
                path
            )
        )

        assert path.endswith(
            ".gz"
        )

        handle.write(
            path
            + "\n"
        )


# =========================================================
# 8. GNU-time parser
# =========================================================

def parse_peak_rss_gib(
    text
):

    match = re.search(
        r"Maximum resident set size \(kbytes\):\s*(\d+)",
        text
    )

    if not match:

        return float(
            "nan"
        )

    return (
        int(
            match.group(1)
        )
        / 1024
        / 1024
    )


# =========================================================
# 9. Check persistent graph checkpoint
# =========================================================

checkpoint_exists = (
    os.path.exists(
        PERSISTENT_GRAPH_PATH
    )
    and
    os.path.exists(
        PERSISTENT_COLOUR_PATH
    )
    and
    os.path.getsize(
        PERSISTENT_GRAPH_PATH
    ) > 0
    and
    os.path.getsize(
        PERSISTENT_COLOUR_PATH
    ) > 0
)


print("=" * 70)
print("CHECKPOINT-SAFE UNITIG WORKFLOW")
print("=" * 70)

print(
    f"Persistent graph checkpoint : "
    f"{checkpoint_exists}"
)

print()


# =========================================================
# 10. STAGE A
#
# Build only when Drive checkpoint is absent.
# =========================================================

graph_built_this_run = False

build_seconds = 0.0

build_peak_rss = float(
    "nan"
)


if checkpoint_exists:

    print(
        "STAGE A: SKIPPED"
    )

    print(
        "Existing Bifrost graph checkpoint found on Drive."
    )

    print()


else:

    print("=" * 70)
    print("STAGE A — BUILD BIFROST GRAPH")
    print("=" * 70)

    build_command = [
        GNU_TIME,
        "-v",
        BIFROST_BIN,
        "build",
        "-t",
        str(
            UNITIG_THREADS
        ),
        "-k",
        str(
            UNITIG_K
        ),
        "-c",
        "-r",
        REFS_PATH,
        "-o",
        LOCAL_GRAPH_PREFIX
    ]


    print(
        " ".join(
            build_command
        )
    )

    print()


    build_start = time.time()


    build_result = subprocess.run(
        build_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )


    build_seconds = (
        time.time()
        - build_start
    )


    with open(
        BUILD_LOG_PATH,
        "w"
    ) as handle:

        handle.write(
            build_result.stdout
        )


    print(
        build_result.stdout
    )


    if build_result.returncode != 0:

        raise RuntimeError(
            "Bifrost graph construction failed. "
            f"See: {BUILD_LOG_PATH}"
        )


    assert os.path.exists(
        LOCAL_GRAPH_PATH
    )


    assert os.path.exists(
        LOCAL_COLOUR_PATH
    )


    assert os.path.getsize(
        LOCAL_GRAPH_PATH
    ) > 0


    assert os.path.getsize(
        LOCAL_COLOUR_PATH
    ) > 0


    build_peak_rss = parse_peak_rss_gib(
        build_result.stdout
    )


    # =====================================================
    # 11. CRITICAL CHECKPOINT:
    # Copy graph to Drive BEFORE Stage B
    # =====================================================

    print()
    print(
        "Saving Bifrost graph checkpoint to Drive..."
    )


    shutil.copy2(
        LOCAL_GRAPH_PATH,
        PERSISTENT_GRAPH_PATH
    )

    shutil.copy2(
        LOCAL_COLOUR_PATH,
        PERSISTENT_COLOUR_PATH
    )


    assert os.path.exists(
        PERSISTENT_GRAPH_PATH
    )


    assert os.path.exists(
        PERSISTENT_COLOUR_PATH
    )


    assert (
        os.path.getsize(
            LOCAL_GRAPH_PATH
        )
        ==
        os.path.getsize(
            PERSISTENT_GRAPH_PATH
        )
    )


    assert (
        os.path.getsize(
            LOCAL_COLOUR_PATH
        )
        ==
        os.path.getsize(
            PERSISTENT_COLOUR_PATH
        )
    )


    graph_checkpoint_gib = (
        (
            os.path.getsize(
                PERSISTENT_GRAPH_PATH
            )
            +
            os.path.getsize(
                PERSISTENT_COLOUR_PATH
            )
        )
        / (1024 ** 3)
    )


    df_build_meta = pd.DataFrame(
        [
            {
                "Pilot Genomes":
                    TEST_N,

                "k-mer Size":
                    UNITIG_K,

                "Threads":
                    UNITIG_THREADS,

                "Build Minutes":
                    build_seconds / 60,

                "Build Peak RSS GiB":
                    build_peak_rss,

                "Graph Checkpoint GiB":
                    graph_checkpoint_gib,

                "Graph Path":
                    PERSISTENT_GRAPH_PATH,

                "Colour Path":
                    PERSISTENT_COLOUR_PATH,

                "Phenotype Used":
                    False
            }
        ]
    )


    df_build_meta.to_csv(
        PERSISTENT_BUILD_META_PATH,
        index=False
    )


    graph_built_this_run = True


    print()
    print(
        "PASS: Stage A graph checkpoint safely persisted to Drive."
    )

    print()


# =========================================================
# 12. Restore graph checkpoint from Drive to local storage
#
# This happens even after a fresh runtime reset.
# =========================================================

print(
    "Restoring graph checkpoint from Drive to local storage..."
)


shutil.copy2(
    PERSISTENT_GRAPH_PATH,
    LOCAL_GRAPH_PATH
)

shutil.copy2(
    PERSISTENT_COLOUR_PATH,
    LOCAL_COLOUR_PATH
)


assert os.path.exists(
    LOCAL_GRAPH_PATH
)


assert os.path.exists(
    LOCAL_COLOUR_PATH
)


graph_checkpoint_gib = (
    (
        os.path.getsize(
            LOCAL_GRAPH_PATH
        )
        +
        os.path.getsize(
            LOCAL_COLOUR_PATH
        )
    )
    / (1024 ** 3)
)


print(
    f"Graph checkpoint size       : "
    f"{graph_checkpoint_gib:.3f} GiB"
)

print()


# =========================================================
# 13. Save graph inventory
# =========================================================

df_graph_inventory = pd.DataFrame(
    [
        {
            "File":
                os.path.basename(
                    LOCAL_GRAPH_PATH
                ),

            "Persistent Path":
                PERSISTENT_GRAPH_PATH,

            "Local Path":
                LOCAL_GRAPH_PATH,

            "Size Bytes":
                os.path.getsize(
                    LOCAL_GRAPH_PATH
                ),

            "Size GiB":
                os.path.getsize(
                    LOCAL_GRAPH_PATH
                )
                / (1024 ** 3)
        },

        {
            "File":
                os.path.basename(
                    LOCAL_COLOUR_PATH
                ),

            "Persistent Path":
                PERSISTENT_COLOUR_PATH,

            "Local Path":
                LOCAL_COLOUR_PATH,

            "Size Bytes":
                os.path.getsize(
                    LOCAL_COLOUR_PATH
                ),

            "Size GiB":
                os.path.getsize(
                    LOCAL_COLOUR_PATH
                )
                / (1024 ** 3)
        }
    ]
)


df_graph_inventory.to_csv(
    GRAPH_INVENTORY_PATH,
    index=False
)


# =========================================================
# 14. STAGE B
# Call unitigs from SAVED graph
# =========================================================

if os.path.exists(
    CALL_FIFO
):

    os.remove(
        CALL_FIFO
    )


if os.path.exists(
    CALL_GZIP
):

    os.remove(
        CALL_GZIP
    )


os.mkfifo(
    CALL_FIFO
)


gzip_process = subprocess.Popen(
    [
        "bash",
        "-c",
        (
            f"gzip -1 -c < "
            f"'{CALL_FIFO}' "
            f"> '{CALL_GZIP}'"
        )
    ]
)


time.sleep(
    1
)


call_command = [
    GNU_TIME,
    "-v",
    UNITIG_CALLER_BIN,
    "--call",
    "--graph",
    LOCAL_GRAPH_PATH,
    "--colours",
    LOCAL_COLOUR_PATH,
    "--out",
    CALL_PREFIX,
    "--kmer",
    str(
        UNITIG_K
    ),
    "--threads",
    str(
        UNITIG_THREADS
    ),
    "--pyseer"
]


print("=" * 70)
print("STAGE B — CALL UNITIGS FROM SAVED GRAPH")
print("=" * 70)

print(
    " ".join(
        call_command
    )
)

print()


call_start = time.time()


call_result = subprocess.run(
    call_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)


call_seconds = (
    time.time()
    - call_start
)


gzip_returncode = (
    gzip_process.wait()
)


with open(
    CALL_LOG_PATH,
    "w"
) as handle:

    handle.write(
        call_result.stdout
    )


print(
    call_result.stdout
)


if call_result.returncode != 0:

    raise RuntimeError(
        "Unitig calling from checkpointed graph failed. "
        f"See: {CALL_LOG_PATH}"
    )


if gzip_returncode != 0:

    raise RuntimeError(
        "Streaming gzip process failed."
    )


if os.path.exists(
    CALL_FIFO
):

    os.remove(
        CALL_FIFO
    )


# =========================================================
# 15. Validate compressed pyseer output
# =========================================================

gzip_test = subprocess.run(
    [
        "gzip",
        "-t",
        CALL_GZIP
    ]
)


assert (
    gzip_test.returncode
    == 0
)


assert os.path.getsize(
    CALL_GZIP
) > 0


call_peak_rss = parse_peak_rss_gib(
    call_result.stdout
)


# =========================================================
# 16. Parse prevalence spectrum
# =========================================================

new_histogram = Counter()

new_unitigs = 0


with gzip.open(
    CALL_GZIP,
    "rt"
) as handle:

    for line in handle:

        line = line.strip()


        if (
            not line
            or "|" not in line
        ):

            continue


        _, calls_part = (
            line.split(
                "|",
                1
            )
        )


        prevalence = len(
            calls_part.strip().split()
        )


        new_histogram[
            prevalence
        ] += 1


        new_unitigs += 1


assert new_unitigs > 0


# =========================================================
# 17. Load original Cell-4 prevalence spectrum
# =========================================================

df_original_spectrum = pd.read_csv(
    PILOT_PREVALENCE_PATH
)


original_histogram = {
    int(
        row[
            "Genomes Present"
        ]
    ):
    int(
        row[
            "Unitigs"
        ]
    )

    for _, row
    in df_original_spectrum.iterrows()
}


original_unitigs = int(
    sum(
        original_histogram.values()
    )
)


unitig_count_match = (
    new_unitigs
    ==
    original_unitigs
)


prevalence_spectrum_match = (
    dict(
        sorted(
            new_histogram.items()
        )
    )
    ==
    dict(
        sorted(
            original_histogram.items()
        )
    )
)


# =========================================================
# 18. Save final validation summary
# =========================================================

compressed_pyseer_gib = (
    os.path.getsize(
        CALL_GZIP
    )
    / (1024 ** 3)
)


df_split_summary = pd.DataFrame(
    [
        {
            "Pilot Genomes":
                TEST_N,

            "k-mer Size":
                UNITIG_K,

            "Threads":
                UNITIG_THREADS,

            "Graph Built This Run":
                graph_built_this_run,

            "Persistent Graph Available":
                True,

            "Build Minutes":
                build_seconds / 60,

            "Build Peak RSS GiB":
                build_peak_rss,

            "Graph Checkpoint GiB":
                graph_checkpoint_gib,

            "Call Minutes":
                call_seconds / 60,

            "Call Peak RSS GiB":
                call_peak_rss,

            "Compressed Pyseer GiB":
                compressed_pyseer_gib,

            "Observed Unitigs":
                new_unitigs,

            "Original Unitigs":
                original_unitigs,

            "Unitig Count Match":
                unitig_count_match,

            "Prevalence Spectrum Match":
                prevalence_spectrum_match,

            "Phenotype Used":
                False
        }
    ]
)


df_split_summary.to_csv(
    SUMMARY_PATH,
    index=False
)


# =========================================================
# 19. Final report
# =========================================================

print()
print("=" * 70)
print("CHECKPOINT-SAFE WORKFLOW VALIDATION")
print("=" * 70)

print(
    f"Graph built this run         : "
    f"{graph_built_this_run}"
)

print(
    f"Persistent graph available   : "
    f"True"
)

print(
    f"Graph checkpoint size        : "
    f"{graph_checkpoint_gib:.3f} GiB"
)

print()

if graph_built_this_run:

    print(
        f"Graph build runtime          : "
        f"{build_seconds / 60:.2f} min"
    )

    print(
        f"Graph build peak RSS         : "
        f"{build_peak_rss:.2f} GiB"
    )

else:

    print(
        "Graph build runtime          : SKIPPED"
    )

    print(
        "Graph build peak RSS         : SKIPPED"
    )


print()

print(
    f"Call-from-graph runtime      : "
    f"{call_seconds / 60:.2f} min"
)

print(
    f"Call-from-graph peak RSS     : "
    f"{call_peak_rss:.2f} GiB"
)

print(
    f"Compressed pyseer output     : "
    f"{compressed_pyseer_gib:.3f} GiB"
)

print()

print(
    f"Observed unitigs             : "
    f"{new_unitigs:,}"
)

print(
    f"Original pilot unitigs       : "
    f"{original_unitigs:,}"
)

print(
    f"Unitig count match           : "
    f"{unitig_count_match}"
)

print(
    f"Prevalence spectrum match    : "
    f"{prevalence_spectrum_match}"
)

print()

print(
    "Persistent graph:"
)

print(
    PERSISTENT_GRAPH_PATH
)

print()

print(
    "Persistent colours:"
)

print(
    PERSISTENT_COLOUR_PATH
)

print()

print(
    "Validation summary:"
)

print(
    SUMMARY_PATH
)

print()


if (
    unitig_count_match
    and prevalence_spectrum_match
):

    print(
        "PASS: The checkpoint-safe Bifrost graph -> "
        "unitig-call workflow reproduces the original "
        "64-genome representation."
    )

else:

    print(
        "REVIEW REQUIRED: The checkpoint-safe workflow "
        "does not fully reproduce the original pilot."
    )

CHECKPOINT-SAFE UNITIG WORKFLOW
Persistent graph checkpoint : False

STAGE A — BUILD BIFROST GRAPH
/usr/bin/time -v /content/unitig_env/bin/Bifrost build -t 2 -k 31 -c -r /content/sequence_variation_work/unitig_split_workflow_test64/refs_gz.txt -o /content/sequence_variation_work/unitig_split_workflow_test64/graph/pilot64_k31_graph

	Command being timed: "/content/unitig_env/bin/Bifrost build -t 2 -k 31 -c -r /content/sequence_variation_work/unitig_split_workflow_test64/refs_gz.txt -o /content/sequence_variation_work/unitig_split_workflow_test64/graph/pilot64_k31_graph"
	User time (seconds): 443.48
	System time (seconds): 3.12
	Percent of CPU this job got: 166%
	Elapsed (wall clock) time (h:mm:ss or m:ss): 4:27.93
	Average shared text size (kbytes): 0
	Average unshared data size (kbytes): 0
	Average stack size (kbytes): 0
	Average total size (kbytes): 0
	Maximum resident set size (kbytes): 407264
	Average resident set size (kbytes): 0
	Major (requiring I/O) page faults: 0
	Minor (recla

KeyboardInterrupt: 

In [ ]:
import os
import subprocess
import shutil

# Kill possible leftover processes from aborted Cell 8
subprocess.run(
    ["pkill", "-f", "unitig-caller"],
    check=False
)

subprocess.run(
    ["pkill", "-f", "Bifrost"],
    check=False
)

# Only remove the abandoned Cell 8 local workspace
CELL8_LOCAL = (
    "/content/sequence_variation_work/"
    "unitig_split_workflow_test64"
)

if os.path.exists(CELL8_LOCAL):
    shutil.rmtree(CELL8_LOCAL)

print("PASS: Cell 8 aborted workflow cleaned up.")
print("Ready for Cell 9.")

PASS: Cell 8 aborted workflow cleaned up.
Ready for Cell 9.


# Cell 9 — Persistent Background Full-Cohort Unitig Calling


In [14]:
import os
import shlex
import shutil
import signal
import subprocess
import time
from datetime import datetime

import pandas as pd


# =========================================================
# FILE 06 — CELL 9
# Full 4,227-genome unitig run
# Persistent / background / Drive-safe version
# =========================================================


# =========================================================
# 1. Configuration
# =========================================================

FULL_N = 4227
UNITIG_K = 31

UNITIG_THREADS = min(
    os.cpu_count() or 1,
    8
)


# =========================================================
# 2. CRITICAL: use DRIVE_ROOT from Cell 1
#
# DO NOT redefine DRIVE_ROOT here.
# =========================================================

assert (
    "DRIVE_ROOT"
    in globals()
), (
    "DRIVE_ROOT is undefined. "
    "Run Cell 1 first."
)


EXPECTED_DRIVE_ROOT = (
    "/content/drive_real/"
    "MyDrive/"
    "AMR-Genome-ML"
)


assert os.path.abspath(
    DRIVE_ROOT
) == EXPECTED_DRIVE_ROOT, (
    "Unsafe or unexpected Drive mount detected.\n"
    f"Current DRIVE_ROOT:\n{DRIVE_ROOT}\n\n"
    f"Expected:\n{EXPECTED_DRIVE_ROOT}\n\n"
    "Run the finalized Cell 1 before Cell 9."
)


assert os.path.isdir(
    DRIVE_ROOT
), (
    f"Project root is unavailable:\n"
    f"{DRIVE_ROOT}"
)


print("=" * 70)
print("CELL 9 — DRIVE SAFETY CHECK")
print("=" * 70)

print(
    f"Verified DRIVE_ROOT : "
    f"{DRIVE_ROOT}"
)

print(
    "PASS: Cell 9 is using the verified real Google Drive."
)

print()


# =========================================================
# 3. Reconstruct File 06 directories from verified root
# =========================================================

SEQUENCE_DIR = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "sequence_variation"
)

SEQUENCE_CHECKPOINT_DIR = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "sequence_variation"
)


os.makedirs(
    SEQUENCE_DIR,
    exist_ok=True
)

os.makedirs(
    SEQUENCE_CHECKPOINT_DIR,
    exist_ok=True
)


# =========================================================
# 4. Tool paths
# =========================================================

UNITIG_ENV = (
    "/content/unitig_env"
)

UNITIG_CALLER_BIN = os.path.join(
    UNITIG_ENV,
    "bin",
    "unitig-caller"
)

GNU_TIME = (
    "/usr/bin/time"
)


# =========================================================
# 5. Restore GNU time automatically
# =========================================================

if not os.path.exists(
    GNU_TIME
):

    print(
        "GNU time missing. Restoring..."
    )

    subprocess.run(
        [
            "apt-get",
            "update",
            "-qq"
        ],
        check=True
    )

    subprocess.run(
        [
            "apt-get",
            "install",
            "-y",
            "-qq",
            "time"
        ],
        check=True
    )


assert os.path.exists(
    GNU_TIME
)


assert os.path.exists(
    UNITIG_CALLER_BIN
), (
    "unitig-caller is missing. "
    "Rerun Cell 3 first."
)


# =========================================================
# 6. Persistent Drive outputs
# =========================================================

FULL_MANIFEST_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_full_k31_input_manifest.csv"
)

FINAL_OUTPUT_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_full_k31.pyseer.gz"
)

PARTIAL_OUTPUT_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_full_k31.pyseer.partial.gz"
)

STATUS_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_full_k31_status.csv"
)

PROGRESS_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_full_k31_progress.csv"
)

PID_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_full_k31_pid.txt"
)

LOG_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_full_k31_run.log"
)

SCRIPT_BACKUP_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_full_k31_background_job.sh"
)


# =========================================================
# 7. Assert every persistent output is on REAL Drive
# =========================================================

persistent_paths = [
    FULL_MANIFEST_PATH,
    FINAL_OUTPUT_PATH,
    PARTIAL_OUTPUT_PATH,
    STATUS_PATH,
    PROGRESS_PATH,
    PID_PATH,
    LOG_PATH,
    SCRIPT_BACKUP_PATH
]


for path in persistent_paths:

    assert path.startswith(
        EXPECTED_DRIVE_ROOT
    ), (
        "Persistent output escaped the verified "
        "Google Drive root:\n"
        f"{path}"
    )


# =========================================================
# 8. Local execution paths
# =========================================================

LOCAL_ROOT = (
    "/content/"
    "sequence_variation_work/"
    "unitig_full_k31_background"
)

LOCAL_OUTPUT_DIR = os.path.join(
    LOCAL_ROOT,
    "output"
)

LOCAL_REFS_PATH = os.path.join(
    LOCAL_ROOT,
    "refs.txt"
)

LOCAL_OUTPUT_PREFIX = os.path.join(
    LOCAL_OUTPUT_DIR,
    "unitig_full_k31"
)

LOCAL_FIFO_PATH = (
    LOCAL_OUTPUT_PREFIX
    + ".pyseer"
)

LOCAL_SCRIPT_PATH = os.path.join(
    LOCAL_ROOT,
    "run_full_unitig.sh"
)


# =========================================================
# 9. Validate cohort restored by Cell 1
# =========================================================

assert (
    "df_sequence_input"
    in globals()
), (
    "df_sequence_input is undefined. "
    "Run Cell 1 first."
)


assert len(
    df_sequence_input
) == FULL_N


assert (
    df_sequence_input[
        "Genome ID"
    ]
    .nunique()
    == FULL_N
)


assert (
    df_sequence_input[
        "FASTA Path"
    ]
    .apply(
        os.path.exists
    )
    .all()
), (
    "At least one FASTA file is unavailable."
)


compressed_count = int(
    df_sequence_input[
        "FASTA Path"
    ]
    .astype(str)
    .str.endswith(
        ".gz"
    )
    .sum()
)


assert compressed_count == FULL_N


# =========================================================
# 10. Ensure FASTA paths also point to real Drive
# =========================================================

bad_fasta_paths = (
    df_sequence_input[
        ~df_sequence_input[
            "FASTA Path"
        ]
        .astype(str)
        .str.startswith(
            "/content/drive_real/MyDrive/"
        )
    ]
)


assert bad_fasta_paths.empty, (
    f"{len(bad_fasta_paths):,} FASTA paths do not "
    "point to the verified real Drive mount."
)


# =========================================================
# 11. Save exact input manifest to real Drive
# =========================================================

df_full_manifest = (
    df_sequence_input[
        [
            "Genome ID",
            "FASTA Path"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


df_full_manifest[
    "Unitig Input Order"
] = range(
    FULL_N
)


assert (
    "Resistant Phenotype"
    not in df_full_manifest.columns
)


df_full_manifest.to_csv(
    FULL_MANIFEST_PATH,
    index=False
)


# =========================================================
# 12. Validate manifest was really written to Drive
# =========================================================

assert os.path.exists(
    FULL_MANIFEST_PATH
)


assert os.path.getsize(
    FULL_MANIFEST_PATH
) > 0


# =========================================================
# 13. Check whether final run is already COMPLETE
# =========================================================

complete_already = False


if os.path.exists(
    FINAL_OUTPUT_PATH
):

    print(
        "Existing final output detected. "
        "Checking gzip integrity..."
    )


    gzip_check = subprocess.run(
        [
            "gzip",
            "-t",
            FINAL_OUTPUT_PATH
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )


    complete_already = (
        gzip_check.returncode
        == 0
        and
        os.path.getsize(
            FINAL_OUTPUT_PATH
        ) > 0
    )


# =========================================================
# 14. Helper: verify stale PID really belongs to our worker
# =========================================================

def pid_matches_unitig_worker(
    pid
):

    proc_cmdline_path = (
        f"/proc/{pid}/cmdline"
    )


    if not os.path.exists(
        proc_cmdline_path
    ):

        return False


    try:

        with open(
            proc_cmdline_path,
            "rb"
        ) as handle:

            cmdline = (
                handle.read()
                .replace(
                    b"\x00",
                    b" "
                )
                .decode(
                    "utf-8",
                    errors="ignore"
                )
            )


        return (
            "run_full_unitig.sh"
            in cmdline
            or
            "unitig_full_k31"
            in cmdline
        )


    except Exception:

        return False


# =========================================================
# 15. Completed output exists -> do nothing
# =========================================================

if complete_already:

    print()
    print("=" * 70)
    print("FULL-COHORT UNITIG RUN")
    print("=" * 70)

    print(
        "A valid completed output already exists:"
    )

    print(
        FINAL_OUTPUT_PATH
    )

    print()

    print(
        "PASS: duplicate full run skipped."
    )


else:

    # =====================================================
    # 16. Detect an already-running REAL job
    # =====================================================

    existing_pid = None

    existing_process_alive = False


    if os.path.exists(
        PID_PATH
    ):

        try:

            with open(
                PID_PATH,
                "r"
            ) as handle:

                existing_pid = int(
                    handle.read().strip()
                )


            os.kill(
                existing_pid,
                0
            )


            existing_process_alive = (
                pid_matches_unitig_worker(
                    existing_pid
                )
            )


        except Exception:

            existing_process_alive = False


    # =====================================================
    # 17. If already running, don't launch duplicate
    # =====================================================

    if existing_process_alive:

        print()
        print("=" * 70)
        print("FULL-COHORT UNITIG RUN")
        print("=" * 70)

        print(
            f"Existing background job PID : "
            f"{existing_pid}"
        )

        print(
            "Status                      : RUNNING"
        )

        print()

        print(
            "No duplicate job was launched."
        )

        print()

        print(
            "Persistent status:"
        )

        print(
            STATUS_PATH
        )

        print()

        print(
            "Progress history:"
        )

        print(
            PROGRESS_PATH
        )


    else:

        # =================================================
        # 18. Remove stale PID record
        # =================================================

        if os.path.exists(
            PID_PATH
        ):

            try:

                os.remove(
                    PID_PATH
                )

            except Exception:

                pass


        # =================================================
        # 19. Archive stale partial output if present
        # =================================================

        if os.path.exists(
            PARTIAL_OUTPUT_PATH
        ):

            stale_bytes = os.path.getsize(
                PARTIAL_OUTPUT_PATH
            )

            stale_gib = (
                stale_bytes
                / (1024 ** 3)
            )


            print(
                f"Stale incomplete output found "
                f"({stale_gib:.3f} GiB)."
            )


            stale_stamp = (
                datetime.now()
                .strftime(
                    "%Y%m%d_%H%M%S"
                )
            )


            stale_path = os.path.join(
                SEQUENCE_CHECKPOINT_DIR,
                (
                    "unitig_full_k31_interrupted_"
                    f"{stale_stamp}.partial.gz"
                )
            )


            shutil.move(
                PARTIAL_OUTPUT_PATH,
                stale_path
            )


            print(
                "Archived interrupted partial output:"
            )

            print(
                stale_path
            )

            print()


        # =================================================
        # 20. Prepare clean LOCAL workspace
        # =================================================

        if os.path.exists(
            LOCAL_ROOT
        ):

            shutil.rmtree(
                LOCAL_ROOT
            )


        os.makedirs(
            LOCAL_OUTPUT_DIR,
            exist_ok=True
        )


        # =================================================
        # 21. refs.txt points directly to .fna.gz on Drive
        # =================================================

        with open(
            LOCAL_REFS_PATH,
            "w"
        ) as handle:

            for path in df_full_manifest[
                "FASTA Path"
            ]:

                path = os.path.abspath(
                    str(
                        path
                    )
                )


                assert os.path.exists(
                    path
                )


                handle.write(
                    path
                    + "\n"
                )


        with open(
            LOCAL_REFS_PATH,
            "r"
        ) as handle:

            refs_count = sum(
                1
                for line in handle
                if line.strip()
            )


        assert refs_count == FULL_N


        # =================================================
        # 22. Shell quoting helper
        # =================================================

        q = shlex.quote


        # =================================================
        # 23. Persistent detached worker script
        # =================================================

        worker_script = f"""#!/usr/bin/env bash

set -u

FULL_N={FULL_N}
KMER={UNITIG_K}
THREADS={UNITIG_THREADS}

UNITIG={q(UNITIG_CALLER_BIN)}
GNU_TIME={q(GNU_TIME)}

REFS={q(LOCAL_REFS_PATH)}
PREFIX={q(LOCAL_OUTPUT_PREFIX)}
FIFO={q(LOCAL_FIFO_PATH)}

PARTIAL={q(PARTIAL_OUTPUT_PATH)}
FINAL={q(FINAL_OUTPUT_PATH)}

STATUS={q(STATUS_PATH)}
PROGRESS={q(PROGRESS_PATH)}
PIDFILE={q(PID_PATH)}
LOG={q(LOG_PATH)}

START_EPOCH=$(date +%s)
START_ISO=$(date -Iseconds)


write_status() {{

    STATUS_VALUE="$1"
    MESSAGE_VALUE="$2"

    NOW_EPOCH=$(date +%s)
    NOW_ISO=$(date -Iseconds)

    ELAPSED_SEC=$((NOW_EPOCH - START_EPOCH))

    ELAPSED_HOURS=$(awk \
        "BEGIN {{printf \\"%.4f\\", $ELAPSED_SEC/3600}}"
    )

    if [ -f "$PARTIAL" ]; then

        BYTES=$(stat -c%s "$PARTIAL" 2>/dev/null || echo 0)

    else

        BYTES=0

    fi


    GIB=$(awk \
        "BEGIN {{printf \\"%.4f\\", $BYTES/1073741824}}"
    )


    printf \
        "Status,Timestamp,Elapsed Hours,Partial GiB,Message\\n" \
        > "$STATUS"


    printf \
        "%s,%s,%s,%s,%s\\n" \
        "$STATUS_VALUE" \
        "$NOW_ISO" \
        "$ELAPSED_HOURS" \
        "$GIB" \
        "$MESSAGE_VALUE" \
        >> "$STATUS"
}}


cleanup_signal() {{

    write_status \
        "INTERRUPTED" \
        "Background worker received termination signal"


    if [ -n "${{UC_PID:-}}" ]; then

        kill "$UC_PID" 2>/dev/null || true

    fi


    if [ -n "${{GZIP_PID:-}}" ]; then

        kill "$GZIP_PID" 2>/dev/null || true

    fi


    rm -f "$FIFO"

    rm -f "$PIDFILE"

    exit 130
}}


trap cleanup_signal INT TERM HUP


mkdir -p {q(LOCAL_OUTPUT_DIR)}


rm -f "$FIFO"

mkfifo "$FIFO"


rm -f "$PARTIAL"


printf \
    "Timestamp,Elapsed Hours,Partial GiB\\n" \
    > "$PROGRESS"


write_status \
    "STARTING" \
    "Preparing gzip stream"


# ---------------------------------------------------------
# Start gzip stream.
# Output is written DIRECTLY to real Google Drive.
# ---------------------------------------------------------

gzip -1 -c < "$FIFO" > "$PARTIAL" &

GZIP_PID=$!


sleep 1


# ---------------------------------------------------------
# Start full cohort unitig-caller
# ---------------------------------------------------------

"$GNU_TIME" -v \
"$UNITIG" \
--call \
--refs "$REFS" \
--out "$PREFIX" \
--kmer "$KMER" \
--threads "$THREADS" \
--pyseer \
> "$LOG" 2>&1 &


UC_PID=$!


# Worker PID, not unitig-caller PID
printf "%s\\n" "$$" > "$PIDFILE"


write_status \
    "RUNNING" \
    "Full cohort graph construction and unitig calling"


NEXT_PROGRESS=$((START_EPOCH + 600))


# ---------------------------------------------------------
# Monitor every minute.
# Persist heartbeat every 10 minutes.
# ---------------------------------------------------------

while kill -0 "$UC_PID" 2>/dev/null
do

    sleep 60


    NOW_EPOCH=$(date +%s)


    # -----------------------------------------------------
    # gzip failure watchdog
    # -----------------------------------------------------

    if ! kill -0 "$GZIP_PID" 2>/dev/null
    then

        echo \
            "ERROR: gzip stream terminated unexpectedly" \
            >> "$LOG"


        kill "$UC_PID" 2>/dev/null || true


        write_status \
            "FAILED" \
            "gzip stream terminated unexpectedly"


        rm -f "$FIFO"

        rm -f "$PIDFILE"

        exit 1

    fi


    # -----------------------------------------------------
    # 10-minute persistent heartbeat
    # -----------------------------------------------------

    if [ "$NOW_EPOCH" -ge "$NEXT_PROGRESS" ]
    then

        ELAPSED_SEC=$((NOW_EPOCH - START_EPOCH))


        ELAPSED_HOURS=$(awk \
            "BEGIN {{printf \\"%.4f\\", $ELAPSED_SEC/3600}}"
        )


        if [ -f "$PARTIAL" ]
        then

            BYTES=$(stat -c%s "$PARTIAL" 2>/dev/null || echo 0)

        else

            BYTES=0

        fi


        GIB=$(awk \
            "BEGIN {{printf \\"%.4f\\", $BYTES/1073741824}}"
        )


        printf \
            "%s,%s,%s\\n" \
            "$(date -Iseconds)" \
            "$ELAPSED_HOURS" \
            "$GIB" \
            >> "$PROGRESS"


        write_status \
            "RUNNING" \
            "Heartbeat"


        NEXT_PROGRESS=$((NEXT_PROGRESS + 600))

    fi

done


# ---------------------------------------------------------
# Collect unitig-caller return code
# ---------------------------------------------------------

wait "$UC_PID"

UC_RC=$?


if [ "$UC_RC" -ne 0 ]
then

    kill "$GZIP_PID" 2>/dev/null || true

    wait "$GZIP_PID" 2>/dev/null || true


    write_status \
        "FAILED" \
        "unitig-caller exited nonzero"


    rm -f "$FIFO"

    rm -f "$PIDFILE"

    exit "$UC_RC"

fi


# ---------------------------------------------------------
# unitig-caller has closed FIFO.
# gzip should now finish normally.
# ---------------------------------------------------------

wait "$GZIP_PID"

GZIP_RC=$?


rm -f "$FIFO"


if [ "$GZIP_RC" -ne 0 ]
then

    write_status \
        "FAILED" \
        "gzip exited nonzero"


    rm -f "$PIDFILE"

    exit "$GZIP_RC"

fi


# ---------------------------------------------------------
# Validate gzip before exposing FINAL file
# ---------------------------------------------------------

write_status \
    "VALIDATING" \
    "Checking gzip integrity"


if ! gzip -t "$PARTIAL"
then

    write_status \
        "FAILED" \
        "gzip integrity check failed"


    rm -f "$PIDFILE"

    exit 1

fi


# ---------------------------------------------------------
# Promote completed output:
# .partial.gz -> .pyseer.gz
# ---------------------------------------------------------

mv -f "$PARTIAL" "$FINAL"


FINAL_BYTES=$(stat -c%s "$FINAL")


FINAL_GIB=$(awk \
    "BEGIN {{printf \\"%.4f\\", $FINAL_BYTES/1073741824}}"
)


END_EPOCH=$(date +%s)

TOTAL_SEC=$((END_EPOCH - START_EPOCH))


TOTAL_HOURS=$(awk \
    "BEGIN {{printf \\"%.4f\\", $TOTAL_SEC/3600}}"
)


printf \
    "Status,Timestamp,Elapsed Hours,Partial GiB,Message\\n" \
    > "$STATUS"


printf \
    "COMPLETE,%s,%s,%s,%s\\n" \
    "$(date -Iseconds)" \
    "$TOTAL_HOURS" \
    "$FINAL_GIB" \
    "Full cohort unitig run completed" \
    >> "$STATUS"


rm -f "$PIDFILE"


exit 0
"""


        # =================================================
        # 24. Save worker locally
        # =================================================

        with open(
            LOCAL_SCRIPT_PATH,
            "w"
        ) as handle:

            handle.write(
                worker_script
            )


        os.chmod(
            LOCAL_SCRIPT_PATH,
            0o755
        )


        # =================================================
        # 25. Persist exact worker script to real Drive
        # =================================================

        shutil.copy2(
            LOCAL_SCRIPT_PATH,
            SCRIPT_BACKUP_PATH
        )


        assert os.path.exists(
            SCRIPT_BACKUP_PATH
        )


        # =================================================
        # 26. Launch detached from notebook kernel
        # =================================================

        background_process = subprocess.Popen(
            [
                "bash",
                LOCAL_SCRIPT_PATH
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            stdin=subprocess.DEVNULL,
            start_new_session=True
        )


        launcher_pid = (
            background_process.pid
        )


        # Give worker enough time to initialize.
        time.sleep(
            5
        )


        process_alive = (
            background_process.poll()
            is None
        )


        assert process_alive, (
            "Background worker terminated immediately. "
            f"Inspect:\n{LOG_PATH}"
        )


        # =================================================
        # 27. Verify status file appears on real Drive
        # =================================================

        status_wait_start = (
            time.time()
        )


        while (
            not os.path.exists(
                STATUS_PATH
            )
            and
            time.time()
            - status_wait_start
            < 30
        ):

            time.sleep(
                1
            )


        assert os.path.exists(
            STATUS_PATH
        ), (
            "Background process started, but persistent "
            "Drive status file was not created."
        )


        # =================================================
        # 28. Verify PID file
        # =================================================

        with open(
            PID_PATH,
            "r"
        ) as handle:

            worker_pid = int(
                handle.read().strip()
            )


        assert worker_pid > 0


        # =================================================
        # 29. Read initial status
        # =================================================

        initial_status = pd.read_csv(
            STATUS_PATH
        )


        assert len(
            initial_status
        ) == 1


        status_value = str(
            initial_status.iloc[0][
                "Status"
            ]
        )


        assert status_value in {
            "STARTING",
            "RUNNING"
        }


        # =================================================
        # 30. Final launcher report
        # =================================================

        print()
        print("=" * 70)
        print("FULL 4,227-GENOME UNITIG JOB LAUNCHED")
        print("=" * 70)

        print(
            f"Genomes                  : "
            f"{FULL_N:,}"
        )

        print(
            f"k-mer size               : "
            f"{UNITIG_K}"
        )

        print(
            f"Threads                  : "
            f"{UNITIG_THREADS}"
        )

        print(
            f"Worker PID               : "
            f"{worker_pid}"
        )

        print(
            f"Process detached          : "
            f"{process_alive}"
        )

        print(
            f"Initial persistent status : "
            f"{status_value}"
        )

        print()

        print(
            "VERIFIED real Drive root:"
        )

        print(
            DRIVE_ROOT
        )

        print()

        print(
            "Partial output is written continuously to REAL Drive:"
        )

        print(
            PARTIAL_OUTPUT_PATH
        )

        print()

        print(
            "Final output will appear only after "
            "successful completion:"
        )

        print(
            FINAL_OUTPUT_PATH
        )

        print()

        print(
            "Persistent status:"
        )

        print(
            STATUS_PATH
        )

        print()

        print(
            "10-minute progress history:"
        )

        print(
            PROGRESS_PATH
        )

        print()

        print(
            "Persistent run log:"
        )

        print(
            LOG_PATH
        )

        print()

        print(
            "PASS: Full-cohort unitig job is running "
            "in the background with all persistent "
            "artifacts on the verified real Google Drive."
        )

CELL 9 — DRIVE SAFETY CHECK
Verified DRIVE_ROOT : /content/drive_real/MyDrive/AMR-Genome-ML
PASS: Cell 9 is using the verified real Google Drive.


FULL 4,227-GENOME UNITIG JOB LAUNCHED
Genomes                  : 4,227
k-mer size               : 31
Threads                  : 2
Worker PID               : 8083
Process detached          : True
Initial persistent status : RUNNING

VERIFIED real Drive root:
/content/drive_real/MyDrive/AMR-Genome-ML

Partial output is written continuously to REAL Drive:
/content/drive_real/MyDrive/AMR-Genome-ML/data/features/sequence_variation/unitig_full_k31.pyseer.partial.gz

Final output will appear only after successful completion:
/content/drive_real/MyDrive/AMR-Genome-ML/data/features/sequence_variation/unitig_full_k31.pyseer.gz

Persistent status:
/content/drive_real/MyDrive/AMR-Genome-ML/checkpoints/sequence_variation/unitig_full_k31_status.csv

10-minute progress history:
/content/drive_real/MyDrive/AMR-Genome-ML/checkpoints/sequence_variation/uniti

In [23]:
import os
import pandas as pd


STATUS_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_full_k31_status.csv"
)

PROGRESS_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_full_k31_progress.csv"
)

PID_PATH = os.path.join(
    SEQUENCE_CHECKPOINT_DIR,
    "unitig_full_k31_pid.txt"
)

PARTIAL_OUTPUT_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_full_k31.pyseer.partial.gz"
)

FINAL_OUTPUT_PATH = os.path.join(
    SEQUENCE_DIR,
    "unitig_full_k31.pyseer.gz"
)


print("=" * 70)
print("FULL UNITIG JOB STATUS")
print("=" * 70)


# ---------------------------------------------------------
# PID / process status
# ---------------------------------------------------------

pid = None
alive = False

if os.path.exists(PID_PATH):

    with open(
        PID_PATH,
        "r"
    ) as handle:

        pid = int(
            handle.read().strip()
        )

    alive = os.path.exists(
        f"/proc/{pid}"
    )


print(
    f"Worker PID        : {pid}"
)

print(
    f"Process alive     : {alive}"
)


# ---------------------------------------------------------
# Persistent status
# ---------------------------------------------------------

if os.path.exists(
    STATUS_PATH
):

    df_status = pd.read_csv(
        STATUS_PATH
    )

    print()

    print(
        "Persistent status:"
    )

    display(
        df_status
    )


# ---------------------------------------------------------
# Partial / final output
# ---------------------------------------------------------

partial_gib = (
    os.path.getsize(
        PARTIAL_OUTPUT_PATH
    )
    / (1024 ** 3)
    if os.path.exists(
        PARTIAL_OUTPUT_PATH
    )
    else 0
)


final_exists = os.path.exists(
    FINAL_OUTPUT_PATH
)


print()

print(
    f"Partial output    : {partial_gib:.4f} GiB"
)

print(
    f"Final output      : {final_exists}"
)


# ---------------------------------------------------------
# Latest heartbeat
# ---------------------------------------------------------

if os.path.exists(
    PROGRESS_PATH
):

    df_progress = pd.read_csv(
        PROGRESS_PATH
    )

    if not df_progress.empty:

        print()

        print(
            "Latest heartbeat:"
        )

        display(
            df_progress.tail(5)
        )


print()

if final_exists:

    print(
        "COMPLETE: final full-cohort output exists."
    )

elif alive:

    print(
        "RUNNING: full-cohort unitig job is still processing."
    )

else:

    print(
        "ATTENTION: worker is no longer running "
        "and final output does not exist."
    )

FULL UNITIG JOB STATUS
Worker PID        : 8083
Process alive     : True

Persistent status:


,Status,Timestamp,Elapsed Hours,Partial GiB,Message
0,RUNNING,2026-09-15T18:50:32+00:00,0.1669,0.0,Heartbeat



Partial output    : 0.0000 GiB
Final output      : False

Latest heartbeat:


,Timestamp,Elapsed Hours,Partial GiB
0,2026-09-15T18:50:32+00:00,0.1669,0.0



RUNNING: full-cohort unitig job is still processing.
